# MCP-Based Pipeline for Clinical Calculators

End-to-end evaluation of the MCP-based framework: calculator selection, parameter extraction and deterministic computation through an MCP registry and server (68 calculators). The notebook is generic: choose the **dataset** (`abacus212`, `abacus409` or `medcalc`) and the **model** in the Configuration cell.

**Pipeline:** Setup -> Selection (CSA) -> Extraction (PEA) -> Computation (CCA) -> Error analysis and category-wise evaluation.

**Requirements:** a CUDA GPU; Python packages listed in `requirements.txt`; the dataset CSV, the MCP registry JSON and the calculator implementations referenced in the Configuration cell.


## Evaluation protocol

Part A is identical in all six notebooks of this release (one MCP notebook and five baseline notebooks). Part B lists what is specific to this pipeline. Both parts describe what the code in this notebook does, so that the numbers in the paper can be checked against it.

### A. Common to all pipelines

**Notation.** N is the number of samples after loading the dataset (rows with missing required fields are dropped). S is the set of samples whose selected calculator equals the ground-truth calculator. An *entity* is one ground-truth input variable, i.e. one key of `Relevant Entities`. S' is the set of samples in S that are scored in the extraction stage, and D the set of samples whose final answer is scored (Part B states both).

**Metrics.**

| Metric | Formula |
|---|---|
| CSA, calculator selection accuracy | CSA = \|S\| / N |
| PEA, parameter extraction accuracy | PEA = (matched entities over S') / (ground-truth entities over S') |
| CCA, calculation correctness accuracy | CCA = \|C\| / \|D\|, C = samples in D whose final answer is correct |

A final answer is correct when the number lies inside the dataset interval `[Lower Limit, Upper Limit]` (inclusive). The category-wise evaluation applies the same three formulas to the samples of one dataset category (`Category` column). CCA uses different denominators in different pipelines (Part B), so it is not directly comparable across pipelines. **No end-to-end metric is computed.**

**Reproducibility settings.**

| Item | Setting |
|---|---|
| Random seed | 42 (Python `random`, NumPy, PyTorch, CUDA) |
| Decoding | greedy; `do_sample=False` in every `generate` call; no temperature, top-p or top-k |
| Quantization | 4-bit NF4 with double quantization, float16 compute dtype (bitsandbytes) |
| Model and dataset | set only in the Configuration cell; nothing else is hard-coded |
| Inputs given to the model | the patient note and the question; ground-truth entities and answers are never part of a prompt |
| Normalization agents | PHONA-S (selection output) and PHONA-E (entity output) are rule-based normalizers, not language models |

**Error taxonomy.** The eight types follow Wang et al. (2025), *From Scores to Steps*. Wang et al. assign them with an LLM judge; this release uses the following deterministic rules, identical in all notebooks (cell *Error taxonomy*). Errors are attributed independently at each stage (not only the first failing stage).

| Type | Counted | Rule |
|---|---|---|
| Formula misselection / hallucination | per sample | the selected calculator differs from the ground truth (N - \|S\| samples, including unmappable outputs) |
| Missing variable | per unmatched entity of S' | no value (empty / null / absent) for the entity |
| Demographic adjustment failure | per unmatched entity | the entity is age, sex / gender, race / ethnicity, pregnancy or BSA |
| Unit conversion error | per unmatched entity | the predicted / ground-truth ratio is within 3 % of a common conversion factor (lb-kg, in-cm, creatinine, glucose, BUN, cholesterol, calcium, bilirubin, factors of 10 and 1000) or the values are related by the Celsius-Fahrenheit formula |
| Clinical misinterpretation | per unmatched entity | the ground-truth value is boolean or categorical (presence, severity, threshold) |
| Incorrect variable extraction | per unmatched entity | any other unmatched numeric value or date |
| Precision error | per sample with a wrong result | the result is outside the dataset interval but within Wang et al.'s rounding tolerance of the ground truth: +-0.05 for answers with at most one decimal place, +-0.005 for two or more |
| Arithmetic error | per sample with a wrong result | any other wrong or non-numeric result |

The rules are tested in this order for extraction errors: missing, demographic, unit conversion, clinical misinterpretation, incorrect extraction. The counts are a diagnostic aid and not metrics; extraction errors are counted per entity, so the total can exceed N.

### B. This pipeline: MCP (proposed)

**Stage 1, selection (CSA).** The model receives the full list of the 68 registered calculators (name and short description, plus dataset-specific disambiguation hints) and returns one calculator name (`max_new_tokens=40`; the prompt is truncated to 4,000 tokens). PHONA-S maps the output to a registry name (alias table, then `difflib` fuzzy matching with cutoff 0.85 for ABACUS212 and 0.80 for ABACUS409; ABACUS212 additionally has deterministic overrides for the Delta Gap / Delta Ratio questions and for Estimated Due Date vs Pregnancy Due Dates). **CSA = |S| / N**; an output that cannot be mapped counts as wrong and stays in the denominator.

**Stage 2, extraction (PEA).** Only samples in S are processed. The model receives the registry schema of the selected calculator (field names, types, units) and returns JSON (`max_new_tokens=700`, prompt truncated to 4,500 tokens). Categorical fields are re-extracted in a second call as a lettered multiple-choice question (`max_new_tokens=5`, prompt truncated to 3,500 tokens). PHONA-E repairs the JSON and maps keys to registry field names. An entity counts as matched when:

- numeric: relative difference of at most 1 % (absolute difference below 0.01 when the ground truth is 0), after unit-aware conversion of the ground truth where the registry defines the required unit;
- dosage fields (drug name plus dose): same drug name and relative dose difference of at most 5 %;
- boolean: equal after converting both sides to true / false;
- categorical: equal after lower-casing, a small synonym table (e.g. *slight* = *mild*, *absent* = *none*), or a boolean answer mapped to the baseline / non-baseline option; the options of the multiple-choice step come from the registry;
- threshold-derived: where the dataset stores a pre-computed boolean for a threshold instead of the raw value (PSI cut-offs; Centor temperature on ABACUS212), the extracted raw value is compared with the threshold and the resulting boolean with the ground truth;
- any other type (e.g. dates): case-insensitive string equality.

S' contains the samples in S whose calculator is not in `STRUCTURAL_MISMATCH_CALCULATORS` (ABACUS212: MME, GAD-7, PHQ-9, Steroid Conversion; ABACUS409: GAD-7); the others are counted in CSA but contribute to neither PEA nor CCA. Ground-truth fields that are outputs of the calculator rather than inputs (`OUTPUT_ONLY_FIELDS`, e.g. *Gestational age* and *EDD* of the Pregnancy Due Dates Calculator) are removed from the ground-truth entities before matching. A sample whose output cannot be parsed contributes its ground-truth entities as unmatched.

**Stage 3, computation (CCA).** D = E, the samples in S' in which *every* ground-truth entity was extracted correctly. Only these are sent to the MCP server, which validates and coerces the entity types against the JSON schema of the calculator and calls the registered Python function (no language-model arithmetic). A numeric result is correct when it lies in `[Lower Limit, Upper Limit]`; a non-numeric result (dates) must equal the ground-truth string. **CCA = |C| / |E|** (`0.0` is printed by the stage cell when E is empty, in which case CCA is undefined rather than zero). Because E is a subset of S, this denominator is smaller than the |S| used by the baseline notebooks. Calculator calls that fail to execute are counted as wrong in CCA but are not assigned to one of the eight error types (they are printed separately).

**Dataset-specific rules.** The disambiguation hints, field hints, aliases, threshold / synonym tables and registry patches are selected by `DATASET_NAME` (cells that begin with `if DATASET_NAME == ...`). They were written for the respective dataset and are part of the evaluated system; for ABACUS409 the MCP server additionally injects a synthetic blood-pressure value for SOFA when only the *hypotension* flag is present. The category-wise evaluation uses the dataset `Category` column; an additional table groups calculators into equation-based and rule-based.


## Setup: environment, files and model

In [ ]:
# Import libraries and verify GPU.
import os, sys, json, re, ast, math, importlib, warnings, random, gc
import numpy as np
import pandas as pd
import torch
from difflib import get_close_matches
from collections import Counter

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('Mem :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 2), 'GB')
else:
    print('WARNING: No CUDA GPU found. This notebook loads a 4-bit quantized model and requires a GPU.')


In [ ]:
# Configuration: the only cell that needs to be edited.
import os

DATASET_NAME      = 'abacus212'                    # 'abacus212', 'abacus409' or 'medcalc'
DATASET_FILE      = 'data/<dataset_file>.csv'      # dataset CSV (see README for the expected columns)
MODEL_ID          = '<hf-model-id>'                # any Hugging Face chat model, e.g. 'org/model-name'
MCP_REGISTRY_FILE = 'registry/mcp_registry_abacus.json'   # MCP registry (68 calculators for ABACUS, 55 for MedCalc-Bench)
CALCULATORS_DIR   = os.path.abspath('calculators') + os.sep   # folder with the calculator .py files
RESULTS_DIR       = f'results/mcp_{DATASET_NAME}_' + MODEL_ID.split('/')[-1]

assert DATASET_NAME in ('abacus212', 'abacus409', 'medcalc'), "DATASET_NAME must be 'abacus212', 'abacus409' or 'medcalc'"
os.makedirs(RESULTS_DIR, exist_ok=True)
for path, label in [(DATASET_FILE, 'Dataset'), (MCP_REGISTRY_FILE, 'Registry'), (CALCULATORS_DIR, 'Calculators')]:
    print(('Found   ' if os.path.exists(path) else 'MISSING '), ':', label, '->', path)


In [ ]:
# Load the dataset and adapt it to the pipeline schema.
if DATASET_NAME == 'abacus212':
    abacus_df = pd.read_csv(DATASET_FILE)
    print('Loaded:', len(abacus_df), 'rows,', len(abacus_df.columns), 'columns')

    RENAME_MAP = {
        'Caprini Score for Venous Thromboembolism': 'Caprini Score for Venous Thromboembolism (2005)',
        'Centor Score for Strep Pharyngitis': 'Centor Score (Modified/McIsaac) for Strep Pharyngitis',
        'Charlson Comorbidity Index': 'Charlson Comorbidity Index (CCI)',
        'Creatinine Clearance': 'Creatinine Clearance (Cockcroft-Gault Equation)',
        'Fibrosis-4 Index for Liver Fibrosis': 'Fibrosis-4 (FIB-4) Index for Liver Fibrosis',
        'Fractional Excretion of Sodium': 'Fractional Excretion of Sodium (FENa)',
        'Free Water Deficit in Hypernatremia': 'Free Water Deficit',
        'Glasgow Coma Score': 'Glasgow Coma Score (GCS)',
        'MELD-Na': 'MELD Na (UNOS/OPTN)',
        'Mean Arterial Pressure': 'Mean Arterial Pressure (MAP)',
        'Morphine Milligram Equivalents Calculator': 'Morphine Milligram Equivalents (MME) Calculator',
        'PSI/PORT Score: Pneumonia Severity Index': 'PSI Score: Pneumonia Severity Index for CAP',
        'Homeostatic Model Assessment for Insulin Resistance': 'HOMA-IR (Homeostatic Model Assessment for Insulin Resistance)',
        'Sequential Organ Failure Assessment Score': 'Sequential Organ Failure Assessment (SOFA) Score',
    }

    def resolve_calculator_name(raw_name):
        name = raw_name.strip()
        if name == 'Corrected QT Interval':
            return 'QTc Bazett Calculator'
        if name == 'Body Mass Index and Body Surface Area':
            return 'Body Mass Index (BMI)'
        if name == 'Ideal Body Weight and Adjusted Body Weight':
            return 'Ideal Body Weight'
        return RENAME_MAP.get(name, name)

    def clean_query_text(text):
        text = re.sub(r'^\s*You\s*\n+', '', text, flags=re.MULTILINE)
        return text.strip()

    abacus_df['Calculator Name'] = abacus_df['Calculator'].apply(resolve_calculator_name)
    abacus_df['Patient Note'] = abacus_df['query'].apply(clean_query_text)
    abacus_df['Question'] = ''

    data_df = abacus_df.rename(columns={
        'relevant_entities': 'Relevant Entities', 'ground_truth': 'Ground Truth Answer',
        'lower_limit': 'Lower Limit', 'upper_limit': 'Upper Limit',
    })[['Calculator Name', 'Category', 'Patient Note', 'Question',
        'Relevant Entities', 'Ground Truth Answer', 'Lower Limit', 'Upper Limit']]

    data_df.dropna(subset=['Calculator Name', 'Patient Note', 'Relevant Entities'], inplace=True)
    data_df.reset_index(drop=True, inplace=True)
    print('Adapted dataset:', len(data_df), 'samples')
    print('Categories:', data_df['Category'].value_counts().to_dict())
elif DATASET_NAME == 'abacus409':
    # Abacus-409 columns: id, vignette, Calculator, ground_truth, Category,
    # relevant_entities, lower_limit, upper_limit. No separate Question column -
    # vignette contains the full case including the question.
    abacus_df = pd.read_csv(DATASET_FILE)
    print('Loaded:', len(abacus_df), 'rows,', len(abacus_df.columns), 'columns')

    RENAME_MAP = {
        'ARISCAT Score': 'ARISCAT Score for Postoperative Pulmonary Complications',
        'Caprini Score for VTE': 'Caprini Score for Venous Thromboembolism (2005)',
        'GAD-7': 'General Anxiety Disorder-7',
        'HAS-BLED Score': 'HAS-BLED Score for Major Bleeding Risk',
        'MELD-Na': 'MELD Na (UNOS/OPTN)',
        'NIH Stroke Scale (NIHSS)': 'NIH Stroke Scale/Score',
        'PSI Score': 'PSI Score: Pneumonia Severity Index for CAP',
        'SOFA Score': 'Sequential Organ Failure Assessment (SOFA) Score',
    }

    def resolve_calculator_name(raw_name):
        return RENAME_MAP.get(raw_name.strip(), raw_name.strip())

    def clean_vignette_text(text):
        text = re.sub(r'^\s*You\s*\n+', '', text, flags=re.MULTILINE)
        return text.strip()

    abacus_df['Calculator Name'] = abacus_df['Calculator'].apply(resolve_calculator_name)
    abacus_df['Patient Note'] = abacus_df['vignette'].apply(clean_vignette_text)
    abacus_df['Question'] = ''  # no separate question field in this dataset

    data_df = abacus_df.rename(columns={
        'relevant_entities': 'Relevant Entities', 'ground_truth': 'Ground Truth Answer',
        'lower_limit': 'Lower Limit', 'upper_limit': 'Upper Limit',
    })[['Calculator Name', 'Category', 'Patient Note', 'Question',
        'Relevant Entities', 'Ground Truth Answer', 'Lower Limit', 'Upper Limit']]

    data_df.dropna(subset=['Calculator Name', 'Patient Note', 'Relevant Entities'], inplace=True)
    data_df.reset_index(drop=True, inplace=True)
    print('Adapted dataset:', len(data_df), 'samples')
    print('Categories:', data_df['Category'].value_counts().to_dict())
    print('Calculators:', data_df['Calculator Name'].value_counts().to_dict())
else:  # medcalc
    data_raw_df = pd.read_csv(DATASET_FILE)
    REQUIRED_COLUMNS = ['Calculator Name', 'Category', 'Patient Note', 'Question',
        'Relevant Entities', 'Ground Truth Answer', 'Lower Limit', 'Upper Limit']
    data_df = data_raw_df[REQUIRED_COLUMNS].copy()
    data_df.dropna(subset=REQUIRED_COLUMNS, inplace=True)
    data_df.reset_index(drop=True, inplace=True)
    print('Cleaned dataset:', len(data_df), 'samples')
    print('Categories:', data_df['Category'].value_counts().to_dict())

In [ ]:
# Load the MCP registry (68 calculators).
with open(MCP_REGISTRY_FILE, 'r') as f:
    MCP_REGISTRY = json.load(f)
print('Registry loaded:', len(MCP_REGISTRY), 'calculators')

missing = set(data_df['Calculator Name'].unique()) - set(MCP_REGISTRY.keys())
print('Dataset calculators missing from registry:', len(missing))
for m in missing:
    print(' ', m)


In [ ]:
# Load the model with 4-bit NF4 quantization (dependencies: see requirements.txt).
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

if os.environ.get('HF_TOKEN'):   # only needed for gated models; never store a token in the notebook
    login(token=os.environ['HF_TOKEN'])

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map={'': 0}, trust_remote_code=True
)
model.eval()
free_mem, total_mem = torch.cuda.mem_get_info(0)
print(MODEL_ID, 'loaded. GPU Free:', round(free_mem/1e9, 2), 'GB /', round(total_mem/1e9, 2), 'GB')


## SECTION 1 — Calculator Selection (CSA)

MCP Tool 1: given a patient note and the full calculator list from the registry, the model selects the single best-matching calculator. Dataset-specific disambiguation hints and alias corrections (selected by `DATASET_NAME`) are added to the prompt and to the PHONA-S normalization step; for ABACUS212 one deterministic override rule is applied (Pregnancy Due Dates Calculator vs Estimated Due Date).


In [ ]:
if DATASET_NAME == 'abacus212':
    # Disambiguation hints (reduce confusion between similarly-named calculators)
    # and aliases (fix truncated/shortened calculator names the SLM sometimes returns).
    DISAMBIGUATION = {
        "Albumin Corrected Anion Gap": "ANION GAP corrected for albumin. Formula: AG+2.5x(4-albumin).",
        "Anion Gap": (
            "Use for basic ANION GAP. Formula: Na - (Cl + HCO3). "
            "IMPORTANT: if the question asks for BOTH 'anion gap' AND 'albumin-corrected anion gap' "
            "together in the same sentence, the correct answer is still just 'Anion Gap'."
        ),
        "Albumin Corrected Delta Gap": "DELTA GAP corrected for albumin.",
        "Albumin Corrected Delta Ratio": "DELTA RATIO corrected for albumin.",
        "Delta Gap": "DELTA GAP simple version. Formula: AG-12.",
        "Delta Ratio": "DELTA RATIO simple version.",
        "Creatinine Clearance (Cockcroft-Gault Equation)": "Cockcroft-Gault. NOT CKD-EPI.",
        "CKD-EPI Equations for Glomerular Filtration Rate": "CKD-EPI 2021. NOT Cockcroft-Gault.",
        "MDRD GFR Equation": "MDRD formula ONLY.",
        "Wells' Criteria for DVT": "LEG symptoms - calf swelling. NOT chest.",
        "Wells' Criteria for Pulmonary Embolism": "CHEST or LUNG symptoms.",
        "Body Mass Index (BMI)": (
            "BMI from height and weight only. DEFAULT CHOICE when question just asks for 'BMI' - "
            "always prefer this over Body Surface Area unless question says 'BSA' explicitly."
        ),
        "Body Surface Area Calculator": (
            "BSA using Mosteller formula. ONLY if question explicitly contains 'body surface area' or 'BSA'."
        ),
        "Glasgow-Blatchford Bleeding Score (GBS)": "GI BLEEDING. NOT Glasgow Coma Score.",
        "Glasgow Coma Score (GCS)": "CONSCIOUSNESS. NOT Glasgow-Blatchford.",
        "HAS-BLED Score for Major Bleeding Risk": "Anticoagulation bleeding risk. NOT GBS.",
        "Estimated Due Date": (
            "WARNING: even if question literally asks 'estimated due date', if the note ALSO mentions "
            "'cycle length' anywhere, you must NOT pick this - pick 'Pregnancy Due Dates Calculator' instead."
        ),
        "Pregnancy Due Dates Calculator": (
            "MANDATORY: if 'cycle length' appears anywhere in the note, this is ALWAYS correct, "
            "regardless of question phrasing."
        ),
        "Estimated Date of Conception": "CONCEPTION DATE only.",
        "Estimated Gestational Age": "HOW FAR ALONG / WEEKS PREGNANT.",
        "MELD Na (UNOS/OPTN)": "Liver disease severity score.",
        "Fibrosis-4 (FIB-4) Index for Liver Fibrosis": "age, AST, ALT, platelets.",
        "Sequential Organ Failure Assessment (SOFA) Score": (
            "ICU organ dysfunction score. NOT SIRS Criteria - if question mentions 'SIRS' or "
            "'sepsis criteria' by name, choose SIRS Criteria instead."
        ),
        "SIRS, Sepsis, and Septic Shock Criteria": (
            "SIRS/sepsis SCREENING criteria (4-criteria checklist). NOT SOFA - if question mentions "
            "'SIRS' or 'sepsis criteria' by name, always choose this over SOFA."
        ),
        "General Anxiety Disorder-7": "GAD-7 anxiety questionnaire.",
        "Patient Health Questionnaire-9": "PHQ-9 depression questionnaire.",
        "Steroid Conversion Calculator": "STEROID drugs only. NOT opioids.",
        "Morphine Milligram Equivalents (MME) Calculator": "OPIOID drugs only. NOT steroids.",
    }

    VALID_NAMES = list(MCP_REGISTRY.keys())

    ALIASES = {
        "Glasgow-Blatchford Bleeding Score": "Glasgow-Blatchford Bleeding Score (GBS)",
        "GBS": "Glasgow-Blatchford Bleeding Score (GBS)",
        "Creatinine Clearance": "Creatinine Clearance (Cockcroft-Gault Equation)",
        "Cockcroft-Gault": "Creatinine Clearance (Cockcroft-Gault Equation)",
        "HEART Score": "HEART Score for Major Cardiac Events",
        "HAS-BLED Score": "HAS-BLED Score for Major Bleeding Risk",
        "HAS-BLED": "HAS-BLED Score for Major Bleeding Risk",
        "HOMA-IR": "HOMA-IR (Homeostatic Model Assessment for Insulin Resistance)",
        "FeverPAIN Score": "FeverPAIN Score for Strep Pharyngitis",
        "SOFA Score": "Sequential Organ Failure Assessment (SOFA) Score",
        "SOFA": "Sequential Organ Failure Assessment (SOFA) Score",
        "SIRS Criteria": "SIRS, Sepsis, and Septic Shock Criteria",
        "SIRS": "SIRS, Sepsis, and Septic Shock Criteria",
        "GAD-7": "General Anxiety Disorder-7",
        "PHQ-9": "Patient Health Questionnaire-9",
        "MELD Na": "MELD Na (UNOS/OPTN)",
        "MELD-Na": "MELD Na (UNOS/OPTN)",
        "Pneumonia Severity Index for CAP": "PSI Score: Pneumonia Severity Index for CAP",
        "PNEUMONIA SEVERITY INDEX FOR CAP": "PSI Score: Pneumonia Severity Index for CAP",
        "pneumonia severity index for cap": "PSI Score: Pneumonia Severity Index for CAP",
        "Morphine Milligram Equivalents": "Morphine Milligram Equivalents (MME) Calculator",
        "CIWA-Ar": "CIWA-Ar for Alcohol Withdrawal",
        "Centor Score (Modified/McIsaac)": "Centor Score (Modified/McIsaac) for Strep Pharyngitis",
        "Centor Score": "Centor Score (Modified/McIsaac) for Strep Pharyngitis",
    }

    print('Selection hints loaded:', len(DISAMBIGUATION), 'disambiguation entries,', len(ALIASES), 'aliases')
elif DATASET_NAME == 'abacus409':
    # Disambiguation hints - only the 10 calculators present in this dataset
    # need focused hints (small option-set reduces ambiguity considerably).
    DISAMBIGUATION = {
        "Caprini Score for Venous Thromboembolism (2005)":
            "VTE/DVT/PE risk score using surgery type, mobility, and many boolean risk factors.",
        "Charlson Comorbidity Index (CCI)":
            "Overall comorbidity burden score using age and multiple disease categories.",
        "HAS-BLED Score for Major Bleeding Risk":
            "Anticoagulation-related MAJOR BLEEDING risk. NOT GBS, NOT Glasgow Coma Score.",
        "MELD Na (UNOS/OPTN)":
            "Liver disease severity score. Uses INR, creatinine, bilirubin, sodium, dialysis status.",
        "NIH Stroke Scale/Score":
            "Stroke severity scale (NIHSS) - sums 15 neurological exam item scores.",
        "PSI Score: Pneumonia Severity Index for CAP":
            "Pneumonia severity/mortality risk score. Uses many vital signs, labs, and comorbidities.",
        "Sequential Organ Failure Assessment (SOFA) Score":
            "ICU organ dysfunction score across respiratory, coagulation, liver, cardiovascular, "
            "neurological, and renal systems.",
        "Wells' Criteria for DVT":
            "LEG symptoms - calf swelling, tenderness. DVT = leg clot. NOT chest/PE.",
        "ARISCAT Score for Postoperative Pulmonary Complications":
            "Postoperative pulmonary complication risk using age, SpO2, surgery type/duration.",
        "General Anxiety Disorder-7":
            "GAD-7 questionnaire - sums 7 anxiety item scores (0-3 each).",
    }

    VALID_NAMES = list(MCP_REGISTRY.keys())

    ALIASES = {
        "Caprini Score": "Caprini Score for Venous Thromboembolism (2005)",
        "Caprini Score for VTE": "Caprini Score for Venous Thromboembolism (2005)",
        "HAS-BLED Score": "HAS-BLED Score for Major Bleeding Risk",
        "HAS-BLED": "HAS-BLED Score for Major Bleeding Risk",
        "MELD-Na": "MELD Na (UNOS/OPTN)",
        "MELD Na": "MELD Na (UNOS/OPTN)",
        "NIHSS": "NIH Stroke Scale/Score",
        "NIH Stroke Scale": "NIH Stroke Scale/Score",
        "NIH Stroke Scale (NIHSS)": "NIH Stroke Scale/Score",
        "PSI Score": "PSI Score: Pneumonia Severity Index for CAP",
        "PSI": "PSI Score: Pneumonia Severity Index for CAP",
        "Pneumonia Severity Index for CAP": "PSI Score: Pneumonia Severity Index for CAP",
        "SOFA Score": "Sequential Organ Failure Assessment (SOFA) Score",
        "SOFA": "Sequential Organ Failure Assessment (SOFA) Score",
        "ARISCAT Score": "ARISCAT Score for Postoperative Pulmonary Complications",
        "ARISCAT": "ARISCAT Score for Postoperative Pulmonary Complications",
        "GAD-7": "General Anxiety Disorder-7",
        "GAD7": "General Anxiety Disorder-7",
        "CCI": "Charlson Comorbidity Index (CCI)",
        "Charlson Comorbidity Index": "Charlson Comorbidity Index (CCI)",
        "Wells Criteria for DVT": "Wells' Criteria for DVT",
        "Wells' Criteria DVT": "Wells' Criteria for DVT",
    }

    print('Selection hints loaded:', len(DISAMBIGUATION), 'disambiguation entries,', len(ALIASES), 'aliases')
else:  # medcalc
    # ============================================================

    from difflib import get_close_matches

    DISAMBIGUATION = {
        "Albumin Corrected Anion Gap":
            "ANION GAP corrected for albumin. Formula: AG+2.5x(4-albumin).",
        "Albumin Corrected Delta Gap":
            "DELTA GAP corrected for albumin. Formula: albumin-corrected AG minus 12.",
        "Albumin Corrected Delta Ratio":
            "DELTA RATIO corrected for albumin. Formula: (albumin-corrected AG-12)/(24-HCO3).",
        "Anion Gap":
            "Basic ANION GAP only. No albumin correction. Formula: Na-(Cl+HCO3).",
        "Delta Gap":
            "DELTA GAP simple version. Formula: AG-12.",
        "Delta Ratio":
            "DELTA RATIO simple version. Formula: (AG-12)/(24-HCO3).",
        "Creatinine Clearance (Cockcroft-Gault Equation)":
            "CREATININE CLEARANCE using Cockcroft-Gault formula. "
            "Use when question says Cockcroft-Gault or CrCl. "
            "Uses age, weight, sex, creatinine. Output: mL/min. "
            "NOT CKD-EPI — completely different formula.",
        "CKD-EPI Equations for Glomerular Filtration Rate":
            "eGFR using CKD-EPI 2021. "
            "Use when question says CKD-EPI or eGFR. "
            "NOT Cockcroft-Gault. Uses age, sex, creatinine. Output: mL/min/1.73m2.",
        "MDRD GFR Equation":
            "GFR using MDRD formula ONLY. NOT CKD-EPI. Uses age, sex, race, creatinine.",
        "Wells' Criteria for DVT":
            "LEG symptoms — calf swelling, tenderness. DVT = leg clot. NOT chest.",
        "Wells' Criteria for Pulmonary Embolism":
            "CHEST or LUNG symptoms — shortness of breath. PE = lung clot. NOT leg only.",
        "Ideal Body Weight":
            "IBW only. Uses height and sex. No actual weight needed.",
        "Adjusted Body Weight":
            "ABW for obese patients. Formula: IBW+0.4x(actual-IBW). NOT same as BMI.",
        "Body Mass Index (BMI)":
            "BMI from height and weight. NOT ideal weight or adjusted weight.",
        "Target weight":
            "Weight for a TARGET BMI value. NOT same as BMI calculation.",
        "Body Surface Area Calculator":
            "BSA in m2. Different from BMI.",
        "Glasgow-Blatchford Bleeding Score (GBS)":
            "GBS for GI/GASTROINTESTINAL BLEEDING risk assessment ONLY. "
            "NOT Glasgow Coma Score, NOT HAS-BLED — these are three separate scores. "
            "GBS = gastrointestinal bleeding severity, uses BUN, hemoglobin, "
            "systolic BP, heart rate, melena, syncope.",
        "Glasgow Coma Score (GCS)":
            "GCS for CONSCIOUSNESS level. NOT Glasgow-Blatchford. GCS = neurological.",
        "HAS-BLED Score for Major Bleeding Risk":
            "HAS-BLED for anticoagulation-related MAJOR BLEEDING risk. "
            "NOT GBS, NOT Glasgow Coma Score. "
            "Uses hypertension, renal/liver disease, stroke, bleeding history, INR, age, drugs, alcohol.",
        "HEART Score for Major Cardiac Events":
            "HEART score for chest pain / major cardiac event risk. "
            "Uses History, ECG, Age, Risk factors, Troponin.",
        "QTc Bazett Calculator":     "ONLY when question says BAZETT formula.",
        "QTc Fridericia Calculator": "ONLY when question says FRIDERICIA formula.",
        "QTc Framingham Calculator": "ONLY when question says FRAMINGHAM formula.",
        "QTc Hodges Calculator":     "ONLY when question says HODGES formula.",
        "QTc Rautaharju Calculator": "ONLY when question says RAUTAHARJU formula.",
        "Estimated Due Date":
            "DELIVERY DATE or DUE DATE (EDD). Uses LMP.",
        "Estimated Date of Conception":
            "CONCEPTION DATE only. Formula: LMP+14 days.",
        "Estimated Gestational Age":
            "HOW FAR ALONG or WEEKS PREGNANT. Output: weeks and days.",
        "MELD Na (UNOS/OPTN)":
            "MELD-Na for liver disease. Uses INR, creatinine, bilirubin, sodium.",
        "Fibrosis-4 (FIB-4) Index for Liver Fibrosis":
            "FIB-4 index. Uses age, AST, ALT, platelet count.",
        "Child-Pugh Score for Cirrhosis Mortality":
            "Child-Pugh for liver cirrhosis. Uses bilirubin, albumin, INR, ascites, encephalopathy.",
        "Centor Score (Modified/McIsaac) for Strep Pharyngitis":
            "Centor or McIsaac score for strep throat.",
        "FeverPAIN Score for Strep Pharyngitis":
            "FeverPAIN score only. NOT Centor.",
        "HOMA-IR (Homeostatic Model Assessment for Insulin Resistance)":
            "HOMA-IR for insulin resistance. Uses fasting glucose and fasting insulin.",
    }

    VALID_NAMES = list(MCP_REGISTRY.keys())

    ALIASES = {
        "Glasgow-Blatchford Bleeding Score":
            "Glasgow-Blatchford Bleeding Score (GBS)",
        "Glasgow Blatchford Bleeding Score":
            "Glasgow-Blatchford Bleeding Score (GBS)",
        "Glasgow Blatchford Score":
            "Glasgow-Blatchford Bleeding Score (GBS)",
        "GBS":
            "Glasgow-Blatchford Bleeding Score (GBS)",
        "Target Weight":
            "Target weight",
        "Creatinine Clearance":
            "Creatinine Clearance (Cockcroft-Gault Equation)",
        "Cockcroft-Gault":
            "Creatinine Clearance (Cockcroft-Gault Equation)",
        "Cockcroft Gault":
            "Creatinine Clearance (Cockcroft-Gault Equation)",
        "Cockcroft-Gault Equation":
            "Creatinine Clearance (Cockcroft-Gault Equation)",
        "Creatinine Clearance (Cockcroft-Gault)":
            "Creatinine Clearance (Cockcroft-Gault Equation)",
        "HEART Score":
            "HEART Score for Major Cardiac Events",
        "HEART score":
            "HEART Score for Major Cardiac Events",
        "HAS-BLED Score":
            "HAS-BLED Score for Major Bleeding Risk",
        "HASBLED Score":
            "HAS-BLED Score for Major Bleeding Risk",
        "HAS-BLED":
            "HAS-BLED Score for Major Bleeding Risk",
        "HOMA-IR":
            "HOMA-IR (Homeostatic Model Assessment for Insulin Resistance)",
        "HOMA IR":
            "HOMA-IR (Homeostatic Model Assessment for Insulin Resistance)",
        "FeverPAIN Score":
            "FeverPAIN Score for Strep Pharyngitis",
        "FeverPAIN score":
            "FeverPAIN Score for Strep Pharyngitis",
    }

In [ ]:
if DATASET_NAME == 'abacus212':
    def mcp_get_calculator_list():
        lines = []
        for name, info in sorted(MCP_REGISTRY.items()):
            cat  = info.get('category', 'general')
            hint = DISAMBIGUATION.get(name, '')
            line = f'- {name}  [{cat}]'
            if hint:
                line += f'  NOTE: {hint}'
            lines.append(line)
        return '\n'.join(lines)

    def postprocess_selection(raw_output, question='', patient_note=''):
        raw = raw_output.strip()
        q_lower = question.lower()
        note_lower = patient_note.lower()

        if 'delta gap' in q_lower and 'albumin' not in q_lower:
            return 'Delta Gap'
        if 'delta ratio' in q_lower and 'albumin' not in q_lower:
            return 'Delta Ratio'

        if raw in ALIASES:
            raw = ALIASES[raw]

        # Deterministic override: SLM consistently mismatches "Estimated Due
        # Date" wording even with strong hints - rule-based correction instead.
        if raw == 'Estimated Due Date' and 'cycle length' in note_lower:
            return 'Pregnancy Due Dates Calculator'

        if raw in VALID_NAMES:
            return raw
        for name in VALID_NAMES:
            if raw.lower() == name.lower():
                return name
        for alias, correct in ALIASES.items():
            if raw.lower() == alias.lower():
                return correct
        matches = get_close_matches(raw, VALID_NAMES, n=1, cutoff=0.85)
        if matches:
            return matches[0]
        return raw

    def mcp_select_calculator(patient_note, question):
        calculator_list = mcp_get_calculator_list()
        system_message = (
            'You are a clinical calculator selection assistant. '
            'You will be provided a list of relevant calculators with descriptions. '
            'Use only this list to select the most appropriate calculator. '
            'Return only the most appropriate calculator name.'
        )
        note_short = patient_note[:1800]
        user_message = (
            f'RELEVANT CALCULATORS:\n{calculator_list}\n\n'
            f'Patient Note: {note_short}\n'
            f'Question: {question}\n'
            f'From the list above, return only the most appropriate calculator name.'
        )
        messages = [{'role': 'system', 'content': system_message}, {'role': 'user', 'content': user_message}]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        # max_length=4000: prevents truncation of the final instruction when
        # the 68-calculator list (with hints) combines with a long patient note.
        inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=4000)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs['input_ids'].shape[-1]
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    print('Selection functions ready.')
elif DATASET_NAME == 'abacus409':
    def mcp_get_calculator_list():
        lines = []
        for name, info in sorted(MCP_REGISTRY.items()):
            cat  = info.get('category', 'general')
            hint = DISAMBIGUATION.get(name, '')
            line = f'- {name}  [{cat}]'
            if hint:
                line += f'  NOTE: {hint}'
            lines.append(line)
        return '\n'.join(lines)

    def postprocess_selection(raw_output, question='', patient_note=''):
        raw = raw_output.strip()
        if raw in ALIASES:
            return ALIASES[raw]
        if raw in VALID_NAMES:
            return raw
        for name in VALID_NAMES:
            if raw.lower() == name.lower():
                return name
        for alias, correct in ALIASES.items():
            if raw.lower() == alias.lower():
                return correct
        matches = get_close_matches(raw, VALID_NAMES, n=1, cutoff=0.80)
        if matches:
            return matches[0]
        return raw

    def mcp_select_calculator(patient_note, question):
        calculator_list = mcp_get_calculator_list()
        system_message = (
            'You are a clinical calculator selection assistant. '
            'You will be provided a list of relevant calculators with descriptions. '
            'Use only this list to select the most appropriate calculator. '
            'Return only the most appropriate calculator name.'
        )
        note_short = patient_note[:1800]
        user_message = (
            f'RELEVANT CALCULATORS:\n{calculator_list}\n\n'
            f'Patient Note: {note_short}\n'
            f'Question: {question}\n'
            f'From the list above, return only the most appropriate calculator name.'
        )
        messages = [{'role': 'system', 'content': system_message}, {'role': 'user', 'content': user_message}]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=4000)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs['input_ids'].shape[-1]
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    print('Selection functions ready.')
else:  # medcalc
    def mcp_get_calculator_list():
        lines = []
        for name, info in sorted(MCP_REGISTRY.items()):
            cat  = info.get('category', 'general')
            hint = DISAMBIGUATION.get(name, '')
            line = f'- {name}  [{cat}]'
            if hint:
                line += f'  NOTE: {hint}'
            else:
                bq = info.get('benchmark_question', '')
                if bq:
                    line += f'  NOTE: {bq[:100]}'
            lines.append(line)
        return '\n'.join(lines)


    def postprocess_selection(raw_output, question='', patient_note=''):
        raw = raw_output.strip()

        q_lower = question.lower()
        if 'delta gap' in q_lower and 'albumin' not in q_lower:
            return 'Delta Gap'
        if 'delta ratio' in q_lower and 'albumin' not in q_lower:
            return 'Delta Ratio'
        if 'albumin corrected delta gap' in q_lower:
            return 'Albumin Corrected Delta Gap'
        if 'albumin corrected delta ratio' in q_lower:
            return 'Albumin Corrected Delta Ratio'

        if raw in ALIASES:
            return ALIASES[raw]

        if raw in VALID_NAMES:
            return raw

        for name in VALID_NAMES:
            if raw.lower() == name.lower():
                return name

        for alias, correct in ALIASES.items():
            if raw.lower() == alias.lower():
                return correct

        matches = get_close_matches(raw, VALID_NAMES, n=1, cutoff=0.90)
        if matches:
            return matches[0]

        return raw


    def mcp_select_calculator(patient_note, question):
        calculator_list = mcp_get_calculator_list()

        system_message = (
            'You are a clinical calculator selection assistant. '
            'You will be provided a list of relevant calculators with '
            'descriptions. Use only this list to select the most '
            'appropriate calculator. '
            'Return only the most appropriate calculator name.'
        )

        note_short = patient_note[:400]

        user_message = (
            f'RELEVANT CALCULATORS:\n{calculator_list}\n\n'
            f'Patient Note: {note_short}\n'
            f'Question: {question}\n'
            f'From the list above, return only the most appropriate '
            f'calculator name.'
        )

        messages = [
            {'role': 'system', 'content': system_message},
            {'role': 'user',   'content': user_message},
        ]

        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = tokenizer(
            prompt_text,
            return_tensors='pt',
            truncation=True,
            max_length=3000
        )
        inputs    = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs['input_ids'].shape[-1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        raw_output = tokenizer.decode(
            outputs[0][input_len:], skip_special_tokens=True
        ).strip()

        return raw_output

    print('Selection functions ready.')

In [ ]:
# Run selection on all samples.
gc.collect(); torch.cuda.empty_cache()

raw_selection_outputs, fixed_selection_outputs = [], []
selection_correct_flags, ground_truth_calculators = [], []

TOTAL = len(data_df)
print(f'CSA: Calculator Selection Started - {TOTAL} samples')

for idx in range(TOTAL):
    row = data_df.iloc[idx]
    patient_note, question, gt_calc = row['Patient Note'], row['Question'], row['Calculator Name']

    raw_out = mcp_select_calculator(patient_note, question)
    fixed_out = postprocess_selection(raw_out, question=(question if DATASET_NAME == 'medcalc' else patient_note), patient_note=patient_note)
    is_correct = (fixed_out.strip() == gt_calc.strip())

    raw_selection_outputs.append(raw_out)
    fixed_selection_outputs.append(fixed_out)
    selection_correct_flags.append(is_correct)
    ground_truth_calculators.append(gt_calc)

    if (idx + 1) % 25 == 0:
        acc = round(sum(selection_correct_flags) / (idx + 1) * 100, 1)
        print(f'Processed: {idx+1}/{TOTAL} | Acc: {acc}%')

print('Selection completed.')


In [ ]:
# Compute CSA.
S_indices = [i for i, c in enumerate(selection_correct_flags) if c]
CSA_accuracy = len(S_indices) / TOTAL * 100

print('=' * 60)
print('CSA: Calculator Selection Accuracy')
print('=' * 60)
print(f'Total Samples          : {TOTAL}')
print(f'Correct Selections |S|  : {len(S_indices)}')
print(f'Wrong Selections        : {TOTAL - len(S_indices)}')
print(f'CSA Accuracy              : {round(CSA_accuracy, 2)} %')
print('=' * 60)

wrong = [(ground_truth_calculators[i], raw_selection_outputs[i], fixed_selection_outputs[i])
         for i, c in enumerate(selection_correct_flags) if not c]
for gt, raw, fixed in wrong:
    print(f'  GT: {gt} | Raw: {raw} | Fixed: {fixed}')

selection_results_df = pd.DataFrame({
    'sample_id': range(TOTAL), 'calculator_name': ground_truth_calculators,
    'category': data_df['Category'].tolist(), 'patient_note': data_df['Patient Note'].tolist(),
    'raw_output': raw_selection_outputs, 'fixed_output': fixed_selection_outputs,
    'selection_correct': selection_correct_flags,
    'lower_limit': data_df['Lower Limit'].tolist(), 'upper_limit': data_df['Upper Limit'].tolist(),
    'ground_truth_answer': data_df['Ground Truth Answer'].tolist(),
    'relevant_entities': data_df['Relevant Entities'].tolist(),
})
SELECTION_CSV = os.path.join(RESULTS_DIR, 'selection_results.csv')
selection_results_df.to_csv(SELECTION_CSV, index=False)
print('Saved:', SELECTION_CSV)


## SECTION 2 — Entity Extraction (PEA)

MCP Tool 2: for each correctly-selected sample, the SLM extracts the required field values from the patient note, guided by the Registry's field schema (type, unit, valid values). Includes: unit-aware comparison, categorical-MCQ debiasing (multiple-choice extraction for categorical fields), threshold-derived matching (PSI, Centor), and GT-key synonym normalization for naming variance between datasets.

In [ ]:
if DATASET_NAME == 'abacus212':
    # Field-specific extraction hints (reinforce correct type/format for
    # fields the SLM tends to get wrong).
    EXTRACTION_FIELD_HINTS = {
        ('Caprini Score for Venous Thromboembolism (2005)', 'Surgery Type'):
            "If the patient had NO surgery at all, use the string 'none' (NOT boolean false).",
        ('Charlson Comorbidity Index (CCI)', 'Liver disease severity'):
            "This is categorical, NOT boolean. Must return the exact string 'none' or 'mild'.",
        ('HAS-BLED Score for Major Bleeding Risk', 'Medication usage predisposing to bleeding'):
            "Only True if note explicitly mentions antiplatelet drugs or NSAIDs.",
        ('Ideal Body Weight', 'weight'):
            "If weight is given in lbs/pounds, convert to kg using: kg = lbs x 0.453592. "
            "Calculate precisely and return the kg value, never the raw lbs number.",
        ('Creatinine Clearance (Cockcroft-Gault Equation)', 'weight'):
            "If weight is given in lbs/pounds, convert to kg using: kg = lbs x 0.453592. "
            "Calculate this conversion precisely - double check your arithmetic.",
    }

    def mcp_get_required_fields(calculator_name):
        if calculator_name not in MCP_REGISTRY:
            return f'ERROR: Calculator not found: {calculator_name}'
        fields = MCP_REGISTRY[calculator_name].get('required_fields', {})
        lines = []
        for display_name, info in fields.items():
            ftype = info.get('type', 'unknown')
            line = f'- "{display_name}": type={ftype}'
            if info.get('unit'):
                line += f', unit={info["unit"]}'
            if info.get('values'):
                line += f', valid values=[{info["values"]}]'
            if info.get('note'):
                line += f'  NOTE: {info["note"]}'
            extra_hint = EXTRACTION_FIELD_HINTS.get((calculator_name, display_name))
            if extra_hint:
                line += f'  HINT: {extra_hint}'
            lines.append(line)
        return '\n'.join(lines)

    print('Field hints loaded:', len(EXTRACTION_FIELD_HINTS))
elif DATASET_NAME == 'abacus409':
    # Field-specific extraction hints.
    EXTRACTION_FIELD_HINTS = {
        ('Caprini Score for Venous Thromboembolism (2005)', 'Surgery Type'):
            "If the patient had NO surgery at all, use the string 'none' (NOT boolean false).",
        ('Charlson Comorbidity Index (CCI)', 'Liver disease severity'):
            "This is categorical, NOT boolean. Must return the exact string 'none' or 'mild'.",
        ('Charlson Comorbidity Index (CCI)', 'Diabetes mellitus'):
            "This is categorical: 'uncomplicated' or 'end organ damage' or 'none'. NOT boolean.",
        ('Charlson Comorbidity Index (CCI)', 'Solid tumor'):
            "This is categorical: 'localized' or 'metastatic' or 'none'. NOT boolean.",
        ('HAS-BLED Score for Major Bleeding Risk', 'Medication usage predisposing to bleeding'):
            "Only True if note explicitly mentions antiplatelet drugs or NSAIDs.",
    }

    def mcp_get_required_fields(calculator_name):
        if calculator_name not in MCP_REGISTRY:
            return f'ERROR: Calculator not found: {calculator_name}'
        fields = MCP_REGISTRY[calculator_name].get('required_fields', {})
        lines = []
        for display_name, info in fields.items():
            ftype = info.get('type', 'unknown')
            line = f'- "{display_name}": type={ftype}'
            if info.get('unit'):
                line += f', unit={info["unit"]}'
            if info.get('values'):
                line += f', valid values=[{info["values"]}]'
            if info.get('note'):
                line += f'  NOTE: {info["note"]}'
            extra_hint = EXTRACTION_FIELD_HINTS.get((calculator_name, display_name))
            if extra_hint:
                line += f'  HINT: {extra_hint}'
            lines.append(line)
        return '\n'.join(lines)

    print('Field hints loaded:', len(EXTRACTION_FIELD_HINTS))
else:  # medcalc
    # ============================================================

    EXTRACTION_FIELD_HINTS = {
        ('APACHE II Score', 'Surgery Type'):
            "Infer from context: urgent/emergent/emergency procedure -> 'Emergency'. "
            "Planned/scheduled procedure -> 'Elective'. "
            "No surgery mentioned at all -> 'Nonoperative'.",
        ('HEART Score for Major Cardiac Events', 'Electrocardiogram Test'):
            "If ECG is not explicitly described as abnormal, or note says ECG was "
            "unremarkable/normal/no ST changes, use 'Normal'. "
            "Only use 'Significant ST deviation' if ST elevation/depression is explicitly stated. "
            "Only use 'Non-specific repolarization disturbance' if explicitly described as such.",
        ('APACHE II Score', 'Temperature'):
            "If the note already states temperature in Fahrenheit (e.g. '102°F' or "
            "'102 degrees Fahrenheit'), output that exact number. Do NOT convert it "
            "to Celsius. Only convert if the note gives Celsius.",
        ('Centor Score (Modified/McIsaac) for Strep Pharyngitis', 'Temperature'):
            "If the note already states temperature in Fahrenheit, output that exact "
            "number. Do NOT convert it to Celsius. Only convert if the note gives Celsius.",
        ('Charlson Comorbidity Index (CCI)', 'Liver disease severity'):
            "This is categorical, not boolean. Must return exact string 'none' or 'mild', "
            "never True/False.",
        ('HAS-BLED Score for Major Bleeding Risk', 'Medication usage predisposing to bleeding'):
            "Only True if note explicitly mentions antiplatelet drugs (aspirin, clopidogrel) "
            "or NSAIDs. Do NOT count anticoagulants like warfarin/heparin, or unrelated "
            "medications, as predisposing medications.",
    }


    def mcp_get_required_fields(calculator_name):
        if calculator_name not in MCP_REGISTRY:
            return f'ERROR: Calculator not found: {calculator_name}'
        fields = MCP_REGISTRY[calculator_name].get('required_fields', {})
        lines  = []
        for display_name, info in fields.items():
            ftype = info.get('type', 'unknown')
            line  = f'- "{display_name}": type={ftype}'
            if info.get('unit'):
                line += f', unit={info["unit"]}'
            if info.get('values'):
                line += f', valid values=[{info["values"]}]'
            if info.get('note'):
                line += f'  NOTE: {info["note"]}'
            extra_hint = EXTRACTION_FIELD_HINTS.get((calculator_name, display_name))
            if extra_hint:
                line += f'  HINT: {extra_hint}'
            lines.append(line)
        return '\n'.join(lines)


    print('Field hints loaded. Total hints:', len(EXTRACTION_FIELD_HINTS))

In [ ]:
if DATASET_NAME == 'abacus212':
    # Registry patches: add fields that specific calculator implementations
    # require but the base registry does not list.
    MCP_REGISTRY['Ideal Body Weight']['required_fields']['weight'] = {
        'display_name': 'weight', 'param_key': 'weight_unused', 'type': 'numeric', 'unit': 'kg',
        'note': 'Present in some GT entries but not required by the IBW formula itself.'
    }
    MCP_REGISTRY['Sequential Organ Failure Assessment (SOFA) Score']['required_fields']['MAP_normal'] = {
        'display_name': 'MAP_normal', 'param_key': 'map_normal', 'type': 'boolean', 'values': 'True or False',
        'note': 'True if the note states mean arterial pressure (MAP) is normal.'
    }
    # Unit fixes applied here (before schema generation) so Computation
    # section doesn't need to patch them again.
    MCP_REGISTRY['QTc Bazett Calculator']['required_fields']['QT Interval']['unit'] = 'sec'
    MCP_REGISTRY['Fibrosis-4 (FIB-4) Index for Liver Fibrosis']['required_fields']['Platelet count']['unit'] = 'µL'

    print('Registry field patches applied: IBW weight, SOFA MAP_normal, QTc unit, Fibrosis-4 unit.')
elif DATASET_NAME == 'abacus409':
    # Registry patches specific to SOFA in this dataset:
    # 1. sofa.py's standalone "hypotension" boolean is functionally inert -
    #    it never triggers a scoring branch even when True (verified via
    #    source inspection). ABACUS409 SOFA samples provide "Hypotension"
    #    directly (not raw BP), so we inject a synthetic low-MAP via
    #    sys_bp/dia_bp at computation time (see Section 3) when needed.
    # 2. Platelet count unit fix (same pattern as Fibrosis-4/ABACUS212 -
    #    the .py's unit converter only accepts 'L','dL','mL','µL' etc,
    #    not '10^9/L').
    MCP_REGISTRY['Sequential Organ Failure Assessment (SOFA) Score']['required_fields']['Platelet count']['unit'] = 'µL'

    print('SOFA Platelet count unit patched (µL).')
else:  # medcalc
    print('No registry patches for MedCalc-Bench.')

In [ ]:
# Extraction output cleaner (handles markdown fences, math-expression bug).
def clean_extraction_output(raw_text):
    text = raw_text.strip()
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    text = text.strip()
    start, end = text.find('{'), text.rfind('}')
    if start != -1 and end != -1:
        text = text[start:end+1]

    def eval_math_expr(match):
        expr = match.group(1)
        try:
            if re.match(r'^[\d\.\s\+\-\*/\(\)]+$', expr):
                return ': ' + str(round(eval(expr), 5))
        except Exception:
            pass
        return match.group(0)

    text = re.sub(r':\s*([\d\.\s]+[\+\-\*/][\d\.\s\+\-\*/\(\)]+)(?=[,\}\n])', eval_math_expr, text)
    try:
        parsed = json.loads(text)
    except Exception:
        try:
            parsed = ast.literal_eval(text)
        except Exception:
            return {}
    if not isinstance(parsed, dict):
        return {}
    fixed = {}
    for key, val in parsed.items():
        if isinstance(val, str):
            v_lower = val.strip().lower()
            if v_lower in ['true', 'yes', 'present', 'positive']:
                val = True
            elif v_lower in ['false', 'no', 'absent', 'negative', 'none', 'null', 'n/a']:
                val = False
            else:
                val = val.strip()
        fixed[key] = val
    return fixed

def parse_numeric(value):
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, list) and len(value) >= 1:
        try:
            return float(value[0])
        except Exception:
            return None
    if isinstance(value, str):
        m = re.search(r'-?\d+\.?\d*', value)
        if m:
            return float(m.group())
    return None

print('Output cleaner ready.')


In [ ]:
if DATASET_NAME == 'medcalc':
    # ============================================================

    UNIT_CONVERSIONS = {
        ('degrees fahrenheit', 'degrees celsius'): lambda f: (f - 32) * 5/9,
        ('degrees celsius', 'degrees fahrenheit'): lambda c: c * 9/5 + 32,
        ('lbs', 'kg'): lambda lb: lb * 0.453592,
        ('kg', 'lbs'): lambda kg: kg / 0.453592,
        ('cm', 'm'): lambda cm: cm / 100,
        ('m', 'cm'): lambda m: m * 100,
        ('in', 'cm'): lambda i: i * 2.54,
        ('cm', 'in'): lambda cm: cm / 2.54,
        ('µmol/l', 'mg/dl'): lambda u: u / 88.42,
        ('mg/dl', 'µmol/l'): lambda m: m * 88.42,
        ('mmol/l', 'mg/dl'): lambda m: m * 18,
        ('mg/dl', 'mmol/l'): lambda m: m / 18,
        ('g/l', 'g/dl'): lambda g: g / 10,
        ('g/dl', 'g/l'): lambda g: g * 10,
        ('mmol/l', 'meq/l'): lambda m: m,
        ('meq/l', 'mmol/l'): lambda m: m,
        ('mm hg', 'mmhg'): lambda x: x,
        ('mmhg', 'mm hg'): lambda x: x,
        ('bpm', 'beats per minute'): lambda x: x,
        ('beats per minute', 'bpm'): lambda x: x,
        ('µl', 'mm^3'): lambda x: x,
        ('mm^3', 'µl'): lambda x: x,
    }


    def convert_gt_to_required_unit(gt_value, calc_name, field_key):
        if not isinstance(gt_value, list) or len(gt_value) != 2:
            return gt_value
        gt_num, gt_unit = gt_value
        if not isinstance(gt_unit, str):
            return gt_value

        required_unit = MCP_REGISTRY.get(calc_name, {}).get('required_fields', {}).get(field_key, {}).get('unit')
        if not required_unit:
            return gt_value

        gt_unit_norm       = gt_unit.strip().lower()
        required_unit_norm = required_unit.strip().lower()

        if gt_unit_norm == required_unit_norm:
            return gt_value

        conv_fn = UNIT_CONVERSIONS.get((gt_unit_norm, required_unit_norm))
        if conv_fn:
            try:
                converted = conv_fn(float(gt_num))
                return [converted, required_unit]
            except Exception:
                return gt_value

        return gt_value


    def values_match(gt_value, pred_value, field_type, calc_name=None, field_key=None):
        if pred_value is None:
            return False

        if field_type == 'numeric':
            if calc_name and field_key:
                gt_value = convert_gt_to_required_unit(gt_value, calc_name, field_key)

            gt_num   = parse_numeric(gt_value)
            pred_num = parse_numeric(pred_value)
            if gt_num is None or pred_num is None:
                return False
            if gt_num == 0:
                return abs(pred_num) < 0.01
            return abs(gt_num - pred_num) / abs(gt_num) <= 0.01

        if field_type == 'special':
            if not (isinstance(gt_value, list) and isinstance(pred_value, list)):
                return False
            if len(gt_value) < 2 or len(pred_value) < 2:
                return False
            name_match = str(gt_value[0]).strip().lower() == str(pred_value[0]).strip().lower()
            gt_dose   = parse_numeric(gt_value[1])
            pred_dose = parse_numeric(pred_value[1])
            if gt_dose is None or pred_dose is None:
                return name_match
            dose_match = abs(gt_dose - pred_dose) / max(abs(gt_dose), 0.01) <= 0.05
            return name_match and dose_match

        if field_type == 'boolean':
            gt_bool   = gt_value if isinstance(gt_value, bool) else str(gt_value).strip().lower() in ['true', 'yes']
            pred_bool = pred_value if isinstance(pred_value, bool) else str(pred_value).strip().lower() in ['true', 'yes']
            return gt_bool == pred_bool

        gt_str   = str(gt_value).strip().lower()
        pred_str = str(pred_value).strip().lower()
        return gt_str == pred_str
else:
    # Unit-aware comparison + categorical fuzzy matching + threshold-derived
    # matching (for GT fields that store a pre-computed boolean instead of
    # the raw value, e.g. PSI's "RR>=30", Centor's "Temperature").
    UNIT_CONVERSIONS = {
        ('degrees fahrenheit', 'degrees celsius'): lambda f: (f - 32) * 5/9,
        ('degrees celsius', 'degrees fahrenheit'): lambda c: c * 9/5 + 32,
        ('lbs', 'kg'): lambda lb: lb * 0.453592,
        ('kg', 'lbs'): lambda kg: kg / 0.453592,
        ('cm', 'm'): lambda cm: cm / 100,
        ('m', 'cm'): lambda m: m * 100,
        ('mmol/l', 'mg/dl'): lambda m: m * 18,
        ('mg/dl', 'mmol/l'): lambda m: m / 18,
        ('mmol/l', 'meq/l'): lambda m: m,
        ('meq/l', 'mmol/l'): lambda m: m,
        ('msec', 'sec'): lambda m: m / 1000,
        ('sec', 'msec'): lambda s: s * 1000,
    }

    def convert_gt_to_required_unit(gt_value, calc_name, field_key):
        if not isinstance(gt_value, list) or len(gt_value) != 2:
            return gt_value
        gt_num, gt_unit = gt_value
        if not isinstance(gt_unit, str):
            return gt_value
        required_unit = MCP_REGISTRY.get(calc_name, {}).get('required_fields', {}).get(field_key, {}).get('unit')
        if not required_unit:
            return gt_value
        gu, ru = gt_unit.strip().lower(), required_unit.strip().lower()
        if gu == ru:
            return gt_value
        conv_fn = UNIT_CONVERSIONS.get((gu, ru))
        if conv_fn:
            try:
                return [conv_fn(float(gt_num)), required_unit]
            except Exception:
                return gt_value
        return gt_value

    CATEGORICAL_SYNONYMS = {'slight': 'mild', 'mild': 'mild', 'absent': 'none', 'none': 'none'}
    BASELINE_WORDS = ['none', 'absent', 'no', 'normal']

    def interpret_boolean_as_categorical(pred_value, enum_values):
        if not isinstance(pred_value, bool):
            return pred_value
        baseline = [v for v in enum_values if v.strip().lower() in BASELINE_WORDS]
        non_baseline = [v for v in enum_values if v.strip().lower() not in BASELINE_WORDS]
        if pred_value is False and baseline:
            return baseline[0]
        if pred_value is True and len(non_baseline) == 1:
            return non_baseline[0]
        return pred_value

    def values_match(gt_value, pred_value, field_type, calc_name=None, field_key=None):
        if pred_value is None:
            return False
        if field_type == 'numeric':
            if calc_name and field_key:
                gt_value = convert_gt_to_required_unit(gt_value, calc_name, field_key)
            gt_num, pred_num = parse_numeric(gt_value), parse_numeric(pred_value)
            if gt_num is None or pred_num is None:
                return False
            # SOFA FiO2: GT sometimes stored as fraction (0.8), extraction as
            # percentage (80) - same value, different scale.
            if calc_name == 'Sequential Organ Failure Assessment (SOFA) Score' and field_key == 'FiO2':
                if gt_num < 1 and pred_num > 1:
                    gt_num = gt_num * 100
            if gt_num == 0:
                return abs(pred_num) < 0.01
            return abs(gt_num - pred_num) / abs(gt_num) <= 0.01
        if field_type == 'special':
            if not (isinstance(gt_value, list) and isinstance(pred_value, list)):
                return False
            if len(gt_value) < 2 or len(pred_value) < 2:
                return False
            name_match = str(gt_value[0]).strip().lower() == str(pred_value[0]).strip().lower()
            gt_dose, pred_dose = parse_numeric(gt_value[1]), parse_numeric(pred_value[1])
            if gt_dose is None or pred_dose is None:
                return name_match
            return name_match and abs(gt_dose - pred_dose) / max(abs(gt_dose), 0.01) <= 0.05
        if field_type == 'boolean':
            def to_bool(v):
                if isinstance(v, bool):
                    return v
                if isinstance(v, (int, float)):
                    return v != 0
                return str(v).strip().lower() in ['true', 'yes', '1']
            return to_bool(gt_value) == to_bool(pred_value)
        if field_type == 'categorical':
            if isinstance(pred_value, bool):
                calc_fields = MCP_REGISTRY.get(calc_name, {}).get('required_fields', {})
                values_str = calc_fields.get(field_key, {}).get('values', '')
                enum_values = [v.strip().strip("'") for v in values_str.split(' or ')] if values_str else []
                pred_value = interpret_boolean_as_categorical(pred_value, enum_values)
            gt_str, pred_str = str(gt_value).strip().lower(), str(pred_value).strip().lower()
            if gt_str == pred_str:
                return True
            gt_norm = CATEGORICAL_SYNONYMS.get(gt_str, gt_str)
            pred_norm = CATEGORICAL_SYNONYMS.get(pred_str, pred_str)
            if gt_norm == pred_norm:
                return True
            if gt_str in ['none', 'no', 'absent'] and any(v in pred_str for v in ['no', 'none', 'absent']):
                return True
            return False
        gt_str, pred_str = str(gt_value).strip().lower(), str(pred_value).strip().lower()
        return gt_str == pred_str

    print('values_match ready.')

In [ ]:
if DATASET_NAME == 'abacus212':
    # Threshold-derived matching: GT sometimes stores a pre-computed boolean
    # (e.g. "RR>=30": True) instead of the raw value - derive the boolean
    # from the extracted raw value for a fair comparison.
    PSI_THRESHOLD_FIELDS = {
        'RR>=30':       {'raw_field': 'respiratory rate',           'threshold': 30,   'op': '>='},
        'Hct<30':       {'raw_field': 'Hematocrit',                 'threshold': 30,   'op': '<'},
        'PaO2<60':      {'raw_field': 'Partial pressure of oxygen', 'threshold': 60,   'op': '<'},
        'pH<7.35':      {'raw_field': 'pH',                          'threshold': 7.35, 'op': '<'},
        'BUN>=30':      {'raw_field': 'Blood Urea Nitrogen (BUN)',  'threshold': 30,   'op': '>='},
        'Glucose>=250': {'raw_field': 'Glucose',                     'threshold': 250,  'op': '>='},
        'Pulse>=125':   {'raw_field': 'Heart Rate or Pulse',        'threshold': 125,  'op': '>='},
        'Na<130':       {'raw_field': 'Sodium',                      'threshold': 130,  'op': '<'},
    }

    def check_threshold(raw_num, threshold, op):
        if raw_num is None:
            return None
        return raw_num >= threshold if op == '>=' else raw_num < threshold

    def match_field_with_threshold_support(gt_key, gt_val, pred_entities, calculator_name, calc_fields):
        if calculator_name == 'PSI Score: Pneumonia Severity Index for CAP' and gt_key in PSI_THRESHOLD_FIELDS:
            spec = PSI_THRESHOLD_FIELDS[gt_key]
            raw_num = parse_numeric(pred_entities.get(spec['raw_field']))
            derived = check_threshold(raw_num, spec['threshold'], spec['op'])
            return derived is not None and derived == bool(gt_val)
        if calculator_name == 'Centor Score (Modified/McIsaac) for Strep Pharyngitis' and gt_key == 'Temperature':
            raw_num = parse_numeric(pred_entities.get('Temperature'))
            if raw_num is not None and raw_num > 50:  # likely Fahrenheit
                raw_num = (raw_num - 32) * 5/9
            derived = check_threshold(raw_num, 38, '>=')
            return derived is not None and derived == bool(gt_val)
        ftype = calc_fields.get(gt_key, {}).get('type', 'string')
        pred_val = pred_entities.get(gt_key)
        return values_match(gt_val, pred_val, ftype, calculator_name, gt_key)

    # GT key synonyms: different naming conventions for the same field
    # across datasets (e.g. SOFA's "Platelets" vs registry's "Platelet count").
    GT_KEY_SYNONYMS = {
        'Sequential Organ Failure Assessment (SOFA) Score': {
            'Platelets': 'Platelet count', 'GCS': 'Glasgow Coma Score',
        },
        'Child-Pugh Score for Cirrhosis Mortality': {
            'INR': 'international normalized ratio',
        },
        'ARISCAT Score for Postoperative Pulmonary Complications': {
            'O2 sat': 'O2 saturation (SpO2)', 'Preoperative anemia': 'Preoperative anemia (Hct<30%)',
            'Emergency': 'Emergency procedure',
        },
    }

    # "Gestational age" and "EDD" are OUTPUTS the calculator computes from
    # LMP+cycle_length - never genuine extraction inputs, exclude from PEA.
    OUTPUT_ONLY_FIELDS = {
        'Pregnancy Due Dates Calculator': {'Gestational age', 'EDD'},
    }

    # Calculators with structurally different GT schema (need dedicated
    # custom logic, not covered by this pipeline) - excluded from PEA/CCA.
    STRUCTURAL_MISMATCH_CALCULATORS = {
        'Morphine Milligram Equivalents (MME) Calculator',
        'General Anxiety Disorder-7',
        'Patient Health Questionnaire-9',
        'Steroid Conversion Calculator',
    }

    def normalize_gt_keys(gt_entities, calculator_name):
        synonym_map = GT_KEY_SYNONYMS.get(calculator_name, {})
        excluded = OUTPUT_ONLY_FIELDS.get(calculator_name, set())
        normalized = {}
        for k, v in gt_entities.items():
            if k in excluded:
                continue
            normalized[synonym_map.get(k, k)] = v
        return normalized

    print('Threshold-matching + GT synonyms ready.')
elif DATASET_NAME == 'abacus409':
    # Threshold-derived matching (PSI stores pre-computed boolean thresholds).
    PSI_THRESHOLD_FIELDS = {
        'RR>=30':       {'raw_field': 'respiratory rate',           'threshold': 30,   'op': '>='},
        'Hct<30':       {'raw_field': 'Hematocrit',                 'threshold': 30,   'op': '<'},
        'PaO2<60':      {'raw_field': 'Partial pressure of oxygen', 'threshold': 60,   'op': '<'},
        'pH<7.35':      {'raw_field': 'pH',                          'threshold': 7.35, 'op': '<'},
        'BUN>=30':      {'raw_field': 'Blood Urea Nitrogen (BUN)',  'threshold': 30,   'op': '>='},
        'Glucose>=250': {'raw_field': 'Glucose',                     'threshold': 250,  'op': '>='},
        'Pulse>=125':   {'raw_field': 'Heart Rate or Pulse',        'threshold': 125,  'op': '>='},
        'Na<130':       {'raw_field': 'Sodium',                      'threshold': 130,  'op': '<'},
    }

    def check_threshold(raw_num, threshold, op):
        if raw_num is None:
            return None
        return raw_num >= threshold if op == '>=' else raw_num < threshold

    def match_field_with_threshold_support(gt_key, gt_val, pred_entities, calculator_name, calc_fields):
        if calculator_name == 'PSI Score: Pneumonia Severity Index for CAP' and gt_key in PSI_THRESHOLD_FIELDS:
            spec = PSI_THRESHOLD_FIELDS[gt_key]
            raw_num = parse_numeric(pred_entities.get(spec['raw_field']))
            derived = check_threshold(raw_num, spec['threshold'], spec['op'])
            return derived is not None and derived == bool(gt_val)
        ftype = calc_fields.get(gt_key, {}).get('type', 'string')
        pred_val = pred_entities.get(gt_key)
        return values_match(gt_val, pred_val, ftype, calculator_name, gt_key)

    # Calculators with structurally different GT schema - excluded from PEA/CCA.
    STRUCTURAL_MISMATCH_CALCULATORS = {
        'General Anxiety Disorder-7',  # GAD-7 uses item-list format in this dataset
    }

    def normalize_gt_keys(gt_entities, calculator_name):
        return gt_entities  # no synonym remapping needed - dataset already aligned

    print('Threshold-matching ready.')
else:  # medcalc
    # MedCalc-Bench ground truth needs no key synonyms, threshold derivation or exclusions.
    STRUCTURAL_MISMATCH_CALCULATORS = set()
    OUTPUT_ONLY_FIELDS = {}

    def normalize_gt_keys(gt_entities, calculator_name):
        return gt_entities

    def match_field_with_threshold_support(gt_key, gt_val, pred_entities, calculator_name, calc_fields):
        ftype = calc_fields.get(gt_key, {}).get('type', 'string')
        return values_match(gt_val, pred_entities.get(gt_key), ftype, calculator_name, gt_key)

    print('Field matching ready.')

In [ ]:
if DATASET_NAME == 'medcalc':
    # ============================================================

    def mcp_extract_entities(calculator_name, patient_note, question):
        required_fields_spec = mcp_get_required_fields(calculator_name)
        system_message = (
            'You are a clinical data extraction assistant. '
            'You will be given the exact input fields required for a clinical '
            'calculator, including field type, unit, and valid values. '
            'Extract ONLY these fields from the patient note. '
            'For boolean fields, if the note says "none of the following: A, B, C" '
            'or "the patient has not had X, Y, Z", set ALL of those fields to False. '
            'For numeric fields, return just the number without units, in the '
            'exact unit shown in the field spec. '
            'CRITICAL: Only convert units if the patient note explicitly states a '
            'DIFFERENT unit than what is required. If the note already uses the '
            'required unit, copy the number exactly as stated - do NOT apply any '
            'conversion factor. For example, if weight is required in kg and the '
            'note says "weight is 72 kg", output 72, NOT 72*0.453592. '
            'Only apply these conversions when the note truly uses a different unit: '
            'weight lbs to kg: multiply by 0.453592. '
            'height cm to m: divide by 100. height inches to cm: multiply by 2.54. '
            'temperature Celsius to Fahrenheit: F = C x 9/5 + 32. '
            'creatinine umol/L to mg/dL: divide by 88.42. '
            'glucose mmol/L to mg/dL: multiply by 18. '
            'albumin g/L to g/dL: divide by 10. '
            'calcium mmol/L to mg/dL: multiply by 4. '
            'mEq/L and mmol/L are the SAME numeric value for sodium, chloride, '
            'bicarbonate, and potassium - do NOT apply any conversion factor '
            'between these two units, just use the same number. '
            'Always CALCULATE THE FINAL NUMBER YOURSELF before writing the JSON. '
            'NEVER write a math expression like "72 * 0.453592" as a JSON value - '
            'only write the final computed number. '
            'For categorical fields, ALWAYS return one of the exact valid values as a '
            'string, even if the field name sounds like a yes/no question '
            '(e.g. "Ascites" is categorical with values absent/slight/moderate, '
            'never return True/False for it). '
            'For date fields, return the date in MM/DD/YYYY format exactly, '
            'inferring the relevant date from context if not explicitly labeled. '
            'For type=special fields with a multi-part value (e.g. steroid name, '
            'dose, unit), return a JSON array with all parts, '
            'e.g. "input steroid": ["Dexamethasone PO", 7.96, "mg"], not just the name. '
            'Use the exact field names shown as JSON keys. '
            'Return only a valid flat JSON object. Do not add explanations.'
        )
        note_full = patient_note
        user_message = (
            f'Calculator: {calculator_name}\n\n'
            f'REQUIRED FIELDS:\n{required_fields_spec}\n\n'
            f'Patient Note:\n{note_full}\n\n'
            f'Question:\n{question}\n\n'
            f'Extract the required fields above from the patient note. '
            f'Return only a valid flat JSON object using the exact field names shown.'
        )
        messages = [
            {'role': 'system', 'content': system_message},
            {'role': 'user',   'content': user_message},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            prompt_text,
            return_tensors='pt',
            truncation=True,
            max_length=4500
        )
        inputs    = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs['input_ids'].shape[-1]
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=700,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        raw_output = tokenizer.decode(
            outputs[0][input_len:], skip_special_tokens=True
        ).strip()
        cleaned = clean_extraction_output(raw_output)
        return raw_output, cleaned
else:
    # Base extraction (Tool 2) + key remapping.
    def mcp_extract_entities(calculator_name, patient_note, question):
        required_fields_spec = mcp_get_required_fields(calculator_name)
        system_message = (
            'You are a clinical data extraction assistant. '
            'You will be given the exact input fields required for a clinical calculator, '
            'including field type, unit, and valid values. Extract ONLY these fields from the patient note. '
            'For boolean fields, if the note says "none of the following: A, B, C" or "the patient has not '
            'had X, Y, Z", set ALL of those fields to False. '
            'For numeric fields, return just the number without units, in the exact unit shown in the field spec. '
            'CRITICAL: Only convert units if the patient note explicitly states a DIFFERENT unit than what is '
            'required. Always CALCULATE THE FINAL NUMBER YOURSELF before writing the JSON. NEVER write a math '
            'expression like "72 * 0.453592" as a JSON value. '
            'For categorical fields, ALWAYS return one of the exact valid values as a string, even if the field '
            'name sounds like a yes/no question. '
            'For date fields, return the date in MM/DD/YYYY format exactly. '
            'For type=special fields with a multi-part value, return a JSON array with all parts. '
            'Use the exact field names shown as JSON keys. Return only a valid flat JSON object. No explanations.'
        )
        user_message = (
            f'Calculator: {calculator_name}\n\nREQUIRED FIELDS:\n{required_fields_spec}\n\n'
            f'Patient Note:\n{patient_note}\n\nQuestion:\n{question}\n\n'
            f'Extract the required fields above from the patient note. '
            f'Return only a valid flat JSON object using the exact field names shown.'
        )
        messages = [{'role': 'system', 'content': system_message}, {'role': 'user', 'content': user_message}]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=4500)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs['input_ids'].shape[-1]
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=700, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        raw_output = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        return raw_output, clean_extraction_output(raw_output)

def remap_extraction_keys(pred_entities, calculator_name):
    calc_fields = MCP_REGISTRY.get(calculator_name, {}).get('required_fields', {})
    registry_keys = list(calc_fields.keys())
    def normalize(s):
        return re.sub(r'[^a-z0-9]', '', s.lower())
    norm_to_registry = {normalize(k): k for k in registry_keys}
    return {norm_to_registry.get(normalize(k), k): v for k, v in pred_entities.items()}

print('Base extraction function ready.')

In [ ]:
if DATASET_NAME == 'medcalc':
    def mcp_extract_entities_final(calculator_name, patient_note, question):
        return mcp_extract_entities(calculator_name, patient_note, question)

    print('Extraction function ready.')
else:
    # Categorical extraction as multiple choice: the model picks one lettered option,
    # which avoids its tendency to answer True/False on categorical fields.
    def extract_categorical_field_mcq(calculator_name, patient_note, question, field_display_name, field_info):
        values_str = field_info.get('values', '')
        enum_values = [v.strip().strip("'") for v in re.split(r"'\s+or\s+'", values_str) if v.strip()]
        if not enum_values:
            return None
        ALL_LETTERS = 'ABCDEFGHIJKLMNOPQRST'
        enum_values = enum_values[:len(ALL_LETTERS)]
        letters = ALL_LETTERS[:len(enum_values)]
        options_text = '\n'.join(f'{letters[i]}. {v}' for i, v in enumerate(enum_values))
        system_message = (
            'You are a clinical data extraction assistant answering a multiple-choice question. '
            'Read the patient note and select EXACTLY ONE letter that best matches the clinical field. '
            'Respond with ONLY the single letter (e.g. "A"). Do not write True, False, Yes, or No - only the letter.'
        )
        user_message = (
            f'Patient Note:\n{patient_note}\n\nField: "{field_display_name}"\nOptions:\n{options_text}\n\n'
            f'Which letter best matches this patient? Answer with only the letter.'
        )
        messages = [{'role': 'system', 'content': system_message}, {'role': 'user', 'content': user_message}]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=3500)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs['input_ids'].shape[-1]
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        for i, letter in enumerate(letters):
            if raw.upper().startswith(letter):
                return enum_values[i]
        return None

    def mcp_extract_entities_final(calculator_name, patient_note, question):
        raw_out, pred_entities = mcp_extract_entities(calculator_name, patient_note, question)
        calc_fields = MCP_REGISTRY.get(calculator_name, {}).get('required_fields', {})
        for display_name, field_info in calc_fields.items():
            if field_info.get('type') == 'categorical':
                mcq_result = extract_categorical_field_mcq(calculator_name, patient_note, question, display_name, field_info)
                if mcq_result is not None:
                    pred_entities[display_name] = mcq_result
        return raw_out, pred_entities

    print('Categorical MCQ extraction ready.')

In [ ]:
# Run extraction on i in S (correctly-selected samples).
extraction_results, extraction_correct_flags = [], []
TOTAL_S = len(S_indices)
print(f'PEA: Entity Extraction Started - {TOTAL_S} samples')

for count, sample_idx in enumerate(S_indices):
    row = data_df.iloc[sample_idx]
    calc_name = ground_truth_calculators[sample_idx]
    patient_note, question = row['Patient Note'], row['Question']

    try:
        gt_entities = ast.literal_eval(row['Relevant Entities'])
    except Exception:
        gt_entities = {}

    if calc_name in STRUCTURAL_MISMATCH_CALCULATORS:
        continue

    gt_entities = normalize_gt_keys(gt_entities, calc_name)
    raw_out, pred_entities = mcp_extract_entities_final(calc_name, patient_note, question)
    pred_entities = remap_extraction_keys(pred_entities, calc_name)

    calc_fields = MCP_REGISTRY.get(calc_name, {}).get('required_fields', {})
    matched, total = 0, len(gt_entities)
    for field_key, gt_val in gt_entities.items():
        if match_field_with_threshold_support(field_key, gt_val, pred_entities, calc_name, calc_fields):
            matched += 1

    is_sample_correct = (matched == total and total > 0)
    extraction_correct_flags.append(is_sample_correct)
    extraction_results.append({
        'sample_id': sample_idx, 'calculator_name': calc_name, 'category': row['Category'],
        'raw_slm_output': raw_out, 'extracted_json': json.dumps(pred_entities),
        'relevant_entities': json.dumps(gt_entities), 'matched_fields': matched,
        'total_gt_fields': total, 'extraction_correct': is_sample_correct,
        'lower_limit': row['Lower Limit'], 'upper_limit': row['Upper Limit'],
        'ground_truth_answer': row['Ground Truth Answer'],
    })
    if (count + 1) % 25 == 0:
        m = sum(r['matched_fields'] for r in extraction_results)
        t = sum(r['total_gt_fields'] for r in extraction_results)
        print(f'Extracted: {count+1}/{TOTAL_S} | Field Acc so far: {round(m/t*100,1) if t else 0}%')

print('Extraction completed:', len(extraction_results), 'samples')
print(f'(Skipped {TOTAL_S - len(extraction_results)} structurally-mismatched samples)')
extraction_df = pd.DataFrame(extraction_results)


In [ ]:
# Compute PEA.
total_matched = sum(r['matched_fields'] for r in extraction_results)
total_gt = sum(r['total_gt_fields'] for r in extraction_results)
PEA_accuracy = (total_matched / total_gt) * 100 if total_gt > 0 else 0.0

print('=' * 60)
print('PEA: Entity Extraction Accuracy')
print('=' * 60)
print(f'Samples evaluated : {len(extraction_results)}')
print(f'Total GT fields   : {total_gt}')
print(f'Total matched     : {total_matched}')
print(f'PEA Accuracy       : {round(PEA_accuracy, 2)} %')
print('=' * 60)

EXTRACTION_CSV = os.path.join(RESULTS_DIR, 'extraction_results.csv')
extraction_df.to_csv(EXTRACTION_CSV, index=False)
print('Saved:', EXTRACTION_CSV)


## SECTION 3 — Computation (CCA)

MCP Server: for every fully-and-correctly-extracted sample, the exact Python calculator function is executed (schema-driven type coercion, enum fuzzy-matching). Formula: `CCA = |C| / |E| x 100`, where E = samples with 100%-correct extraction. Includes a graceful fallback for CURB-65's Diastolic BP (clinically only Systolic OR Diastolic is required, but the .py implementation hard-requires both).

In [ ]:
# Auto-generate JSON Schema for every calculator.
def generate_json_schema(calculator_name):
    fields = MCP_REGISTRY.get(calculator_name, {}).get('required_fields', {})
    properties, required = {}, []
    for display_name, info in fields.items():
        ftype = info.get('type', 'unknown')
        prop = {}
        if ftype == 'numeric':
            prop['type'] = 'number'
            if info.get('unit'):
                prop['description'] = f"unit: {info['unit']}"
        elif ftype == 'boolean':
            prop['type'] = 'boolean'
        elif ftype == 'categorical':
            prop['type'] = 'string'
            values_str = info.get('values', '')
            enum_vals = []
            if values_str:
                for part in re.split(r"'\s+or\s+'", values_str):
                    cleaned = part.strip().strip("'").strip()
                    if cleaned:
                        enum_vals.append(cleaned)
            if enum_vals:
                prop['enum'] = enum_vals
        elif ftype == 'date':
            prop['type'] = 'string'; prop['format'] = 'date'
        elif ftype == 'special':
            prop['type'] = 'array'; prop['items'] = {}
        else:
            prop['type'] = 'string'
        if info.get('note') and 'description' not in prop:
            prop['description'] = info['note']
        properties[display_name] = prop
        required.append(display_name)
    return {'type': 'object', 'properties': properties, 'required': required}

JSON_SCHEMA_REGISTRY = {c: generate_json_schema(c) for c in MCP_REGISTRY.keys()}
print('JSON Schema generated for', len(JSON_SCHEMA_REGISTRY), 'calculators.')


In [ ]:
if DATASET_NAME == 'abacus212':
    # MCP Server: schema-driven type coercion + value aliases + CURB-65
    # Diastolic BP fallback (graceful handling of a real .py limitation).
    if CALCULATORS_DIR not in sys.path:
        sys.path.insert(0, CALCULATORS_DIR)

    VALUE_ALIASES = {
        'Glasgow Coma Score (GCS)': {'Best motor response': {'localizing pain': 'localizes pain'}},
        'HEART Score for Major Cardiac Events': {
            'Initial troponin': {
                '1-3 times normal limit': 'between the normal limit or up to three times the normal limit',
                '1-3x normal limit': 'between the normal limit or up to three times the normal limit',
            }
        },
    }

    def apply_value_alias(calculator_name, display_name, value):
        calc_aliases = VALUE_ALIASES.get(calculator_name, {})
        field_aliases = calc_aliases.get(display_name, {})
        if isinstance(value, str):
            return field_aliases.get(value.strip().lower(), field_aliases.get(value, value))
        return value

    def coerce_value(raw_value, schema_prop, calculator_name=None, field_key=None):
        if raw_value is None:
            return None
        if calculator_name and field_key and isinstance(raw_value, str):
            raw_value = apply_value_alias(calculator_name, field_key, raw_value)
        jtype = schema_prop.get('type', 'string')
        if jtype == 'number':
            return parse_numeric(raw_value)
        if jtype == 'boolean':
            if isinstance(raw_value, bool):
                return raw_value
            return str(raw_value).strip().lower() in ['true', 'yes', '1']
        if jtype == 'array':
            return raw_value if isinstance(raw_value, list) else [raw_value]
        val = str(raw_value).strip()
        enum_vals = schema_prop.get('enum')
        if enum_vals:
            for ev in enum_vals:
                if val.lower() == ev.lower():
                    return ev
            matches = get_close_matches(val, enum_vals, n=1, cutoff=0.6)
            if matches:
                return matches[0]
            lower_map = {ev.lower(): ev for ev in enum_vals}
            matches = get_close_matches(val.lower(), list(lower_map.keys()), n=1, cutoff=0.5)
            if matches:
                return lower_map[matches[0]]
        return val

    class MCPServer:
        def __init__(self, registry, schema_registry, calculators_dir):
            self.registry, self.schema_registry, self.calculators_dir = registry, schema_registry, calculators_dir
        def list_tools(self):
            return [{'name': c, 'inputSchema': self.schema_registry.get(c, {})} for c in self.registry.keys()]
        def call_tool(self, tool_name, arguments):
            schema = self.schema_registry.get(tool_name)
            if schema is None:
                return self._error_response(f'Unknown tool: {tool_name}')
            calc_info = self.registry.get(tool_name, {})
            file_path, func_name = calc_info.get('file_path', ''), calc_info.get('function_name', '')
            fields = calc_info.get('required_fields', {})
            if not file_path or not func_name:
                return self._error_response('Missing file_path or function_name')
            coerced_params, missing = {}, []
            for display_name, schema_prop in schema['properties'].items():
                raw_val = arguments.get(display_name)
                # Graceful fallback: CURB-65's .py hard-requires Diastolic BP
                # even though clinically only Systolic OR Diastolic matters.
                if raw_val is None and tool_name == 'CURB-65 Score for Pneumonia Severity' and display_name == 'Diastolic Blood Pressure':
                    if arguments.get('Systolic Blood Pressure') is not None:
                        raw_val = 70
                if raw_val is None:
                    missing.append(display_name); continue
                coerced = coerce_value(raw_val, schema_prop, tool_name, display_name)
                if coerced is None:
                    missing.append(display_name); continue
                field_info = fields.get(display_name, {})
                param_key, unit, ftype = field_info.get('param_key', display_name), field_info.get('unit'), field_info.get('type')
                coerced_params[param_key] = (coerced, unit) if (ftype == 'numeric' and unit) else coerced
            try:
                module_name = file_path.replace('.py', '')
                calc_module = importlib.import_module(module_name)
                calc_func = getattr(calc_module, func_name)
                result = calc_func(coerced_params)
                answer = result.get('Answer', result.get('answer', None)) if isinstance(result, dict) else (result[0] if isinstance(result, (list, tuple)) else result)
                return self._success_response(answer, missing)
            except Exception as e:
                return self._error_response(f'{type(e).__name__}: {str(e)}', missing)
        def _success_response(self, answer, missing):
            return {'jsonrpc': '2.0', 'result': {'answer': answer, 'missing_fields': missing}, 'error': None}
        def _error_response(self, message, missing=None):
            return {'jsonrpc': '2.0', 'result': None, 'error': {'message': message, 'missing_fields': missing or []}}

    mcp_server = MCPServer(MCP_REGISTRY, JSON_SCHEMA_REGISTRY, CALCULATORS_DIR)
    print('MCP Server initialized with', len(mcp_server.list_tools()), 'tools.')
elif DATASET_NAME == 'abacus409':
    # MCP Server: schema-driven type coercion + value aliases + CURB-65
    # Diastolic BP fallback + SOFA Hypotension->MAP fallback.
    if CALCULATORS_DIR not in sys.path:
        sys.path.insert(0, CALCULATORS_DIR)

    VALUE_ALIASES = {
        'Glasgow Coma Score (GCS)': {'Best motor response': {'localizing pain': 'localizes pain'}},
        'HEART Score for Major Cardiac Events': {
            'Initial troponin': {
                '1-3 times normal limit': 'between the normal limit or up to three times the normal limit',
                '1-3x normal limit': 'between the normal limit or up to three times the normal limit',
            }
        },
    }

    def apply_value_alias(calculator_name, display_name, value):
        calc_aliases = VALUE_ALIASES.get(calculator_name, {})
        field_aliases = calc_aliases.get(display_name, {})
        if isinstance(value, str):
            return field_aliases.get(value.strip().lower(), field_aliases.get(value, value))
        return value

    def coerce_value(raw_value, schema_prop, calculator_name=None, field_key=None):
        if raw_value is None:
            return None
        if calculator_name and field_key and isinstance(raw_value, str):
            raw_value = apply_value_alias(calculator_name, field_key, raw_value)
        jtype = schema_prop.get('type', 'string')
        if jtype == 'number':
            return parse_numeric(raw_value)
        if jtype == 'boolean':
            if isinstance(raw_value, bool):
                return raw_value
            return str(raw_value).strip().lower() in ['true', 'yes', '1']
        if jtype == 'array':
            return raw_value if isinstance(raw_value, list) else [raw_value]
        val = str(raw_value).strip()
        enum_vals = schema_prop.get('enum')
        if enum_vals:
            for ev in enum_vals:
                if val.lower() == ev.lower():
                    return ev
            matches = get_close_matches(val, enum_vals, n=1, cutoff=0.6)
            if matches:
                return matches[0]
            lower_map = {ev.lower(): ev for ev in enum_vals}
            matches = get_close_matches(val.lower(), list(lower_map.keys()), n=1, cutoff=0.5)
            if matches:
                return lower_map[matches[0]]
        return val

    class MCPServer:
        def __init__(self, registry, schema_registry, calculators_dir):
            self.registry, self.schema_registry, self.calculators_dir = registry, schema_registry, calculators_dir
        def list_tools(self):
            return [{'name': c, 'inputSchema': self.schema_registry.get(c, {})} for c in self.registry.keys()]
        def call_tool(self, tool_name, arguments):
            schema = self.schema_registry.get(tool_name)
            if schema is None:
                return self._error_response(f'Unknown tool: {tool_name}')
            calc_info = self.registry.get(tool_name, {})
            file_path, func_name = calc_info.get('file_path', ''), calc_info.get('function_name', '')
            fields = calc_info.get('required_fields', {})
            if not file_path or not func_name:
                return self._error_response('Missing file_path or function_name')
            coerced_params, missing = {}, []
            for display_name, schema_prop in schema['properties'].items():
                raw_val = arguments.get(display_name)
                if raw_val is None and tool_name == 'CURB-65 Score for Pneumonia Severity' and display_name == 'Diastolic Blood Pressure':
                    if arguments.get('Systolic Blood Pressure') is not None:
                        raw_val = 70
                if raw_val is None:
                    missing.append(display_name); continue
                coerced = coerce_value(raw_val, schema_prop, tool_name, display_name)
                if coerced is None:
                    missing.append(display_name); continue
                field_info = fields.get(display_name, {})
                param_key, unit, ftype = field_info.get('param_key', display_name), field_info.get('unit'), field_info.get('type')
                coerced_params[param_key] = (coerced, unit) if (ftype == 'numeric' and unit) else coerced

            # SOFA fix: standalone "hypotension" boolean never scores points
            # in sofa.py (verified via source) - inject synthetic low-MAP.
            if tool_name == 'Sequential Organ Failure Assessment (SOFA) Score':
                if coerced_params.get('hypotension') is True and 'sys_bp' not in coerced_params:
                    coerced_params['sys_bp'] = (90, 'mm Hg')
                    coerced_params['dia_bp'] = (50, 'mm Hg')

            try:
                module_name = file_path.replace('.py', '')
                calc_module = importlib.import_module(module_name)
                calc_func = getattr(calc_module, func_name)
                result = calc_func(coerced_params)
                answer = result.get('Answer', result.get('answer', None)) if isinstance(result, dict) else (result[0] if isinstance(result, (list, tuple)) else result)
                return self._success_response(answer, missing)
            except Exception as e:
                return self._error_response(f'{type(e).__name__}: {str(e)}', missing)
        def _success_response(self, answer, missing):
            return {'jsonrpc': '2.0', 'result': {'answer': answer, 'missing_fields': missing}, 'error': None}
        def _error_response(self, message, missing=None):
            return {'jsonrpc': '2.0', 'result': None, 'error': {'message': message, 'missing_fields': missing or []}}

    mcp_server = MCPServer(MCP_REGISTRY, JSON_SCHEMA_REGISTRY, CALCULATORS_DIR)
    print('MCP Server initialized with', len(mcp_server.list_tools()), 'tools.')
else:  # medcalc
    # ============================================================


    if CALCULATORS_DIR not in sys.path:
        sys.path.insert(0, CALCULATORS_DIR)


    def coerce_value(raw_value, schema_prop):
        if raw_value is None:
            return None
        jtype = schema_prop.get('type', 'string')

        if jtype == 'number':
            return parse_numeric(raw_value)

        if jtype == 'boolean':
            if isinstance(raw_value, bool):
                return raw_value
            return str(raw_value).strip().lower() in ['true', 'yes', '1']

        if jtype == 'array':
            if isinstance(raw_value, list):
                return raw_value
            return [raw_value]

        val = str(raw_value).strip()
        enum_vals = schema_prop.get('enum')
        if enum_vals:
            for ev in enum_vals:
                if val.lower() == ev.lower():
                    return ev
            matches = get_close_matches(val, enum_vals, n=1, cutoff=0.6)
            if matches:
                return matches[0]
            lower_map = {ev.lower(): ev for ev in enum_vals}
            matches = get_close_matches(val.lower(), list(lower_map.keys()), n=1, cutoff=0.5)
            if matches:
                return lower_map[matches[0]]

        return val


    class MCPServer:
        def __init__(self, registry, schema_registry, calculators_dir):
            self.registry        = registry
            self.schema_registry = schema_registry
            self.calculators_dir = calculators_dir

        def list_tools(self):
            tools = []
            for calc_name in self.registry.keys():
                tools.append({
                    'name'       : calc_name,
                    'description': self.registry[calc_name].get('benchmark_question', ''),
                    'inputSchema': self.schema_registry.get(calc_name, {}),
                })
            return tools

        def call_tool(self, tool_name, arguments):
            request = {'jsonrpc': '2.0', 'method': 'tools/call', 'params': {'name': tool_name, 'arguments': arguments}}

            schema = self.schema_registry.get(tool_name)
            if schema is None:
                return self._error_response(request, f'Unknown tool: {tool_name}')

            calc_info = self.registry.get(tool_name, {})
            file_path = calc_info.get('file_path', '')
            func_name = calc_info.get('function_name', '')
            fields    = calc_info.get('required_fields', {})

            if not file_path or not func_name:
                return self._error_response(request, 'Missing file_path or function_name')

            coerced_params = {}
            missing        = []
            for display_name, schema_prop in schema['properties'].items():
                raw_val = arguments.get(display_name)
                if raw_val is None:
                    missing.append(display_name)
                    continue
                coerced = coerce_value(raw_val, schema_prop)
                if coerced is None:
                    missing.append(display_name)
                    continue

                field_info = fields.get(display_name, {})
                param_key  = field_info.get('param_key', display_name)
                unit       = field_info.get('unit')
                ftype      = field_info.get('type')

                if ftype == 'numeric' and unit:
                    coerced_params[param_key] = (coerced, unit)
                else:
                    coerced_params[param_key] = coerced

            try:
                module_name = file_path.replace('.py', '')
                calc_module = importlib.import_module(module_name)
                calc_func   = getattr(calc_module, func_name)
                result      = calc_func(coerced_params)

                if isinstance(result, dict):
                    answer = result.get('Answer', result.get('answer', None))
                elif isinstance(result, (list, tuple)):
                    answer = result[0] if len(result) > 0 else None
                else:
                    answer = result

                return self._success_response(request, answer, missing)

            except Exception as e:
                return self._error_response(request, f'{type(e).__name__}: {str(e)}', missing)

        def _success_response(self, request, answer, missing):
            return {'jsonrpc': '2.0', 'result': {'answer': answer, 'missing_fields': missing}, 'error': None}

        def _error_response(self, request, message, missing=None):
            return {'jsonrpc': '2.0', 'result': None, 'error': {'message': message, 'missing_fields': missing or []}}


    mcp_server = MCPServer(MCP_REGISTRY, JSON_SCHEMA_REGISTRY, CALCULATORS_DIR)
    tools = mcp_server.list_tools()
    print(f'MCP Server re-initialized with {len(tools)} tools.')

In [ ]:
if DATASET_NAME == 'abacus212':
    # Pre-conversion: scale/unit fixes applied to extracted values before they
    # are passed to the calculator function (the extraction itself is unchanged).
    fully_extracted = [r for r in extraction_results if r['extraction_correct']]

    for r in fully_extracted:
        if r['calculator_name'] == 'QTc Bazett Calculator':
            ent = json.loads(r['extracted_json'])
            if ent.get('QT Interval') is not None and ent['QT Interval'] > 10:
                ent['QT Interval'] = ent['QT Interval'] / 1000
                r['extracted_json'] = json.dumps(ent)
        if r['calculator_name'] == 'Fibrosis-4 (FIB-4) Index for Liver Fibrosis':
            ent = json.loads(r['extracted_json'])
            if ent.get('Platelet count') is not None and ent['Platelet count'] < 1000:
                ent['Platelet count'] = ent['Platelet count'] * 1000
                r['extracted_json'] = json.dumps(ent)

    print('Pre-conversions applied. Fully extracted (E):', len(fully_extracted))
elif DATASET_NAME == 'abacus409':
    # Pre-conversion: SOFA Platelet count needs x1000 scale-fix (same
    # pattern verified for Fibrosis-4/ABACUS212 - raw extracted values
    # are correct, only the unit-scale passed to the .py needs fixing).
    fully_extracted = [r for r in extraction_results if r['extraction_correct']]

    for r in fully_extracted:
        if r['calculator_name'] == 'Sequential Organ Failure Assessment (SOFA) Score':
            ent = json.loads(r['extracted_json'])
            if ent.get('Platelet count') is not None and ent['Platelet count'] < 1000:
                ent['Platelet count'] = ent['Platelet count'] * 1000
                r['extracted_json'] = json.dumps(ent)

    print('Pre-conversions applied. Fully extracted (E):', len(fully_extracted))
else:  # medcalc
    fully_extracted = [r for r in extraction_results if r['extraction_correct']]
    print('Fully extracted (E):', len(fully_extracted))

In [ ]:
# Run computation on all fully-extracted samples.
computation_results = []
TOTAL_COMP = len(fully_extracted)
print(f'CCA: Computation Started - {TOTAL_COMP} samples')

for count, ext_row in enumerate(fully_extracted):
    calc_name = ext_row['calculator_name']
    entities = json.loads(ext_row['extracted_json'])
    response = mcp_server.call_tool(calc_name, entities)
    predicted_answer = response['result']['answer'] if response['result'] else None
    exec_error = response['error']['message'] if response['error'] else None
    missing = (response['result'] or response['error'] or {}).get('missing_fields', [])
    lower, upper = ext_row['lower_limit'], ext_row['upper_limit']
    try:
        pred_num = float(predicted_answer)
        is_correct = (float(lower) <= pred_num <= float(upper))
    except Exception:
        is_correct = (str(predicted_answer).strip() == str(ext_row['ground_truth_answer']).strip())
    computation_results.append({
        'sample_id': ext_row['sample_id'], 'calculator_name': calc_name, 'category': ext_row['category'],
        'extracted_json': ext_row['extracted_json'], 'predicted_answer': predicted_answer,
        'execution_error': exec_error, 'missing_fields': ';'.join(missing) if missing else '',
        'ground_truth_answer': ext_row['ground_truth_answer'], 'lower_limit': lower, 'upper_limit': upper,
        'computation_correct': is_correct,
    })
    if (count + 1) % 25 == 0:
        print(f'Computed: {count+1}/{TOTAL_COMP} | Correct: {sum(r["computation_correct"] for r in computation_results)}')

print('Computation completed.')
computation_df = pd.DataFrame(computation_results)


In [ ]:
# Compute CCA.
N_E = len(fully_extracted)
N_C = sum(r['computation_correct'] for r in computation_results)
exec_errors = [r for r in computation_results if r['execution_error']]
CCA_accuracy = (N_C / N_E) * 100 if N_E > 0 else 0.0

print('=' * 60)
print('CCA: Computation Accuracy')
print('=' * 60)
print(f'Fully Extracted (E)       : {N_E}')
print(f'Correct Computations |C|  : {N_C}')
print(f'Execution Errors          : {len(exec_errors)}')
print(f'CCA Accuracy (|C|/|E|)     : {round(CCA_accuracy, 2)} %')
print('=' * 60)

if exec_errors:
    print('\nExecution errors:')
    seen = set()
    for r in exec_errors:
        key = (r['calculator_name'], r['execution_error'][:80])
        if key not in seen:
            seen.add(key)
            print(f"  [{r['calculator_name']}] {r['execution_error'][:120]}")

COMPUTATION_CSV = os.path.join(RESULTS_DIR, 'computation_results.csv')
computation_df.to_csv(COMPUTATION_CSV, index=False)
print('Saved:', COMPUTATION_CSV)

print()
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'CSA (Selection)   : {round(CSA_accuracy, 2)}%')
print(f'PEA (Extraction)  : {round(PEA_accuracy, 2)}%')
print(f'CCA (Computation) : {round(CCA_accuracy, 2)}%')
print('=' * 60)


## SECTION 4 — Error Analysis + Category-Wise Evaluation

Classifies every remaining computation error into a standard error taxonomy, and breaks down CSA/PEA/CCA performance by both computational nature (Equation-based vs Rule-based) and clinical category (Lab, Physical, Date, Dosage, Risk, Severity, Diagnosis).

## Error analysis, category-wise evaluation and summary

The next three cells are identical in all notebooks except for the first one, which collects the per-sample results of this pipeline.

In [ ]:
# Per-sample evaluation table (inputs of the error analysis and of the category-wise evaluation).
import json
import pandas as pd

sel = list(selection_correct_flags)
ext_info, ext_unmatched = {}, []
for r in extraction_results:
    gt_ent = json.loads(r['relevant_entities'])
    try:
        pred_ent = json.loads(r['extracted_json'])
    except Exception:
        pred_ent = {}
    calc_fields = MCP_REGISTRY.get(r['calculator_name'], {}).get('required_fields', {})
    for key, gt_val in gt_ent.items():
        if not match_field_with_threshold_support(key, gt_val, pred_ent, r['calculator_name'], calc_fields):
            ext_unmatched.append({"sample_id": r['sample_id'], "category": data_df.iloc[r['sample_id']]["Category"],
                                  "key": key, "gt": gt_val, "pred": pred_ent.get(key)})
    ext_info[r['sample_id']] = (r['total_gt_fields'], r['matched_fields'])

comp_info, comp_wrong, technical_errors = {}, [], 0
for r in computation_results:
    comp_info[r['sample_id']] = bool(r['computation_correct'])
    if not r['computation_correct']:
        if r['execution_error']:
            technical_errors += 1   # calculator call failed to execute: wrong in CCA, reported as a technical error, not one of the eight types
        else:
            comp_wrong.append({"sample_id": r['sample_id'], "category": data_df.iloc[r['sample_id']]["Category"],
                               "pred": r['predicted_answer'], "gt": r['ground_truth_answer']})
print('Technical errors (calculator call failed to execute; counted wrong in CCA, not one of the eight types):', technical_errors)
eval_df = pd.DataFrame([{
    "sample_id": i, "category": data_df.iloc[i]["Category"], "calculator": data_df.iloc[i]["Calculator Name"],
    "selected_correct": bool(sel[i]),
    "ext_evaluated": i in ext_info, "n_entities": ext_info.get(i, (0, 0))[0], "n_matched": ext_info.get(i, (0, 0))[1],
    "comp_evaluated": i in comp_info, "comp_correct": bool(comp_info.get(i, False)),
} for i in range(len(data_df))])

In [ ]:
# Error taxonomy: rule-based operationalization of the categories of Wang et al. (2025), "From Scores to Steps".
# The same rules are used in all six notebooks.
import re

ERROR_TYPES = [
    "formula_misselection_hallucination",  # selected calculator differs from the ground-truth calculator
    "incorrect_variable_extraction",       # wrong value of a numeric variable (or date)
    "missing_variable",                    # required variable absent from the extraction
    "clinical_misinterpretation",          # wrong boolean / categorical value (presence, severity, threshold)
    "demographic_adjustment_failure",      # wrong or missing age, sex, race, BSA or pregnancy variable
    "arithmetic_error",                    # formula and inputs right, result wrong beyond rounding
    "unit_conversion_error",               # value differs from the ground truth by a unit-conversion factor
    "precision_error",                     # result within the rounding tolerance of the ground truth but outside the dataset interval
]
DEMOGRAPHIC_TOKENS = {"age", "sex", "gender", "race", "ethnicity", "pregnancy", "pregnant", "bsa"}
UNIT_FACTORS = [2.20462, 0.45359, 2.54, 0.3937, 88.4, 0.01131, 18.0182, 0.0555, 2.8, 0.357,
                38.67, 0.02586, 4.008, 0.2495, 17.1, 0.0585, 10.0, 0.1, 1000.0, 0.001]
UNIT_FACTOR_TOLERANCE = 0.03    # relative tolerance when comparing the value ratio with a conversion factor
ROUNDING_TOLERANCE = {1: 0.05, 2: 0.005}   # Wang et al.: +-0.05 for one decimal place, +-0.005 for two
DATE_PATTERN = re.compile(r"^\s*\d{1,2}/\d{1,2}/\d{2,4}")

def scalar(v):
    return v[0] if isinstance(v, (list, tuple)) and len(v) > 0 else v

def to_float(v):
    v = scalar(v)
    if v is None or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip().replace(",", "")
    if DATE_PATTERN.match(s):
        return None
    m = re.search(r"-?\d+\.?\d*", s)
    return float(m.group(0)) if m else None

def is_missing(v):
    return v is None or str(v).strip().lower() in ("", "none", "null", "nan")

def is_demographic_key(key):
    tokens = set(re.split(r"[^a-z0-9]+", str(key).lower()))
    return bool(tokens & DEMOGRAPHIC_TOKENS) or "body_surface" in str(key).lower() or "surface area" in str(key).lower()

def is_unit_error(gt, pred):
    g, p = to_float(gt), to_float(pred)
    if g is None or p is None or g == 0 or p == 0:
        return False
    ratio = p / g
    if any(abs(ratio / f - 1) <= UNIT_FACTOR_TOLERANCE for f in UNIT_FACTORS):
        return True
    return abs(p - (g * 9 / 5 + 32)) <= 1 or abs(p - (g - 32) * 5 / 9) <= 1   # Celsius <-> Fahrenheit

def classify_extraction_error(key, gt, pred):
    if is_missing(scalar(pred)):
        return "missing_variable"
    if is_demographic_key(key):
        return "demographic_adjustment_failure"
    if is_unit_error(gt, pred):
        return "unit_conversion_error"
    g = scalar(gt)
    if to_float(g) is None and not DATE_PATTERN.match(str(g)):
        return "clinical_misinterpretation"
    return "incorrect_variable_extraction"

def decimals(text):
    s = str(scalar(text)).strip()
    m = re.search(r"\.(\d+)", s)
    return min(len(m.group(1)), 2) if m else 0

def classify_computation_error(pred, gt):
    p, g = to_float(pred), to_float(gt)
    if p is None or g is None:
        return "arithmetic_error"
    tol = ROUNDING_TOLERANCE.get(decimals(pred), ROUNDING_TOLERANCE[1] if decimals(pred) == 0 else 0.005)
    return "precision_error" if abs(p - g) <= tol else "arithmetic_error"

In [ ]:
# Error analysis, category-wise evaluation and overall summary (identical in all notebooks).
# Inputs built by the previous cell: eval_df (one row per sample), ext_unmatched, comp_wrong.
import json, os
import pandas as pd

N = len(eval_df)
n_selected = int(eval_df["selected_correct"].sum())
ext_part = eval_df[eval_df["ext_evaluated"]]
comp_part = eval_df[eval_df["comp_evaluated"]]

def pct(a, b):
    return round(a / b * 100, 2) if b else float("nan")

# ---- overall metrics
CSA = pct(n_selected, N)                                   # |S| / N
PEA = pct(int(ext_part["n_matched"].sum()), int(ext_part["n_entities"].sum()))   # matched / ground-truth entities
CCA = pct(int(comp_part["comp_correct"].sum()), len(comp_part))                  # |C| / |D|

print("=" * 70)
print(f"CSA = |S| / N                : {n_selected} / {N} = {CSA} %")
print(f"PEA = matched / entities     : {int(ext_part['n_matched'].sum())} / {int(ext_part['n_entities'].sum())} = {PEA} %")
print(f"CCA = |C| / |D|              : {int(comp_part['comp_correct'].sum())} / {len(comp_part)} = {CCA} %")
print("=" * 70)

# ---- error analysis
error_counts = {t: 0 for t in ERROR_TYPES}
error_records = []
for _, r in eval_df[~eval_df["selected_correct"]].iterrows():
    error_counts["formula_misselection_hallucination"] += 1
    error_records.append({"sample_id": r["sample_id"], "category": r["category"], "stage": "selection",
                          "error_type": "formula_misselection_hallucination"})
for e in ext_unmatched:
    t = classify_extraction_error(e["key"], e["gt"], e["pred"])
    error_counts[t] += 1
    error_records.append({"sample_id": e["sample_id"], "category": e["category"], "stage": "extraction",
                          "field": e["key"], "error_type": t})
for w in comp_wrong:
    t = classify_computation_error(w["pred"], w["gt"])
    error_counts[t] += 1
    error_records.append({"sample_id": w["sample_id"], "category": w["category"], "stage": "computation",
                          "error_type": t})

total_errors = sum(error_counts.values())
print(f'{"Error type":<40} {"Count":>7} {"Share":>8}')
for t in ERROR_TYPES:
    print(f"{t:<40} {error_counts[t]:>7} {pct(error_counts[t], total_errors):>7}%")
print(f'{"TOTAL":<40} {total_errors:>7}')
print("(formula errors and computation errors are counted per sample, extraction errors per unmatched entity)")

# ---- category-wise evaluation (same formulas as above, restricted to one category)
def stage_table(group_col):
    rows = []
    for g in sorted(eval_df[group_col].dropna().unique(), key=str):
        d = eval_df[eval_df[group_col] == g]
        e = d[d["ext_evaluated"]]
        c = d[d["comp_evaluated"]]
        rows.append({group_col: g, "N": len(d), "S": int(d["selected_correct"].sum()),
                     "CSA_%": pct(int(d["selected_correct"].sum()), len(d)),
                     "PEA_%": pct(int(e["n_matched"].sum()), int(e["n_entities"].sum())),
                     "CCA_%": pct(int(c["comp_correct"].sum()), len(c)),
                     "D": len(c)})
    return pd.DataFrame(rows)

cat_df = stage_table("category")
print(cat_df.to_string(index=False))
if "EQUATION_BASED_CALCS" in globals():
    eval_df["nature"] = eval_df["calculator"].apply(lambda c: "Equation" if c in EQUATION_BASED_CALCS else "Rule-based")
    nature_df = stage_table("nature")
    print(nature_df.to_string(index=False))
    nature_df.to_csv(os.path.join(RESULTS_DIR, "nature_wise_results.csv"), index=False)

# ---- save
os.makedirs(RESULTS_DIR, exist_ok=True)
pd.DataFrame(error_records).to_csv(os.path.join(RESULTS_DIR, "error_analysis.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "error_summary.json"), "w") as f:
    json.dump({"error_counts": error_counts, "total_errors": total_errors}, f, indent=4)
cat_df.to_csv(os.path.join(RESULTS_DIR, "category_results.csv"), index=False)
eval_df.to_csv(os.path.join(RESULTS_DIR, "per_sample_evaluation.csv"), index=False)
summary = {"dataset": DATASET_NAME, "model": MODEL_ID, "N": N,
           "S": n_selected, "CSA_%": CSA,
           "PEA_matched": int(ext_part["n_matched"].sum()), "PEA_entities": int(ext_part["n_entities"].sum()), "PEA_%": PEA,
           "CCA_correct": int(comp_part["comp_correct"].sum()), "CCA_denominator": len(comp_part), "CCA_%": CCA}
if "technical_errors" in globals():
    summary["technical_errors"] = int(technical_errors)   # MCP only
with open(os.path.join(RESULTS_DIR, "overall_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)
pd.DataFrame([summary]).to_csv(os.path.join(RESULTS_DIR, "All_combined_results.csv"), index=False)
print("Results saved in", RESULTS_DIR)

In [ ]:
# Calculator-wise PEA breakdown (ABACUS409 only).
if DATASET_NAME == 'abacus409':
    # Calculator-wise PEA breakdown.
    calc_m2_stats = {}
    for r in extraction_results:
        calc = r['calculator_name']
        if calc not in calc_m2_stats:
            calc_m2_stats[calc] = {'matched': 0, 'total': 0, 'samples': 0, 'fully_correct': 0}
        calc_m2_stats[calc]['matched'] += r['matched_fields']
        calc_m2_stats[calc]['total'] += r['total_gt_fields']
        calc_m2_stats[calc]['samples'] += 1
        if r['extraction_correct']:
            calc_m2_stats[calc]['fully_correct'] += 1

    rows = []
    for calc, stats in calc_m2_stats.items():
        field_acc = round(stats['matched'] / stats['total'] * 100, 1) if stats['total'] > 0 else 0
        full_acc = round(stats['fully_correct'] / stats['samples'] * 100, 1)
        rows.append({
            'Calculator': calc, 'Category': MCP_REGISTRY.get(calc, {}).get('category', 'unknown'),
            'N_Fields': len(MCP_REGISTRY.get(calc, {}).get('required_fields', {})),
            'Samples': stats['samples'], 'Field_Acc%': field_acc, 'Fully_Correct%': full_acc,
        })

    m2_breakdown_df = pd.DataFrame(rows).sort_values('Field_Acc%', ascending=True)
    print('=' * 100)
    print('PEA CALCULATOR-WISE BREAKDOWN (worst to best)')
    print('=' * 100)
    print(m2_breakdown_df.to_string(index=False))

    PEA_BREAKDOWN_CSV = os.path.join(RESULTS_DIR, 'm2_calculator_breakdown.csv')
    m2_breakdown_df.to_csv(PEA_BREAKDOWN_CSV, index=False)
    print('\nSaved:', PEA_BREAKDOWN_CSV)

## Save results

In [ ]:
# List the result files written during this run.
print('Result files saved in', RESULTS_DIR)
for f in sorted(os.listdir(RESULTS_DIR)):
    print(' ', f)
